In [ ]:
print("Ranjeet-100 epochs")

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

class BEN14KDataset(Dataset):
    def __init__(self, data_dir, parquet_file, split='train'):
        """
        data_dir: Path to the extracted BEN_14k directory
        parquet_file: Path to metadata.parquet
        split: 'train', 'validation', or 'test'
        """
        self.data_dir = data_dir
        self.split = split
        
        print(f"Loading {split} metadata...")
        df = pd.read_parquet(parquet_file)
        df = df[df['split'] == split]
        
        s1_folder = os.path.join(data_dir, 'BigEarthNet-S1', split)
        s2_folder = os.path.join(data_dir, 'BigEarthNet-S2', split)
        
        # --- BULLETPROOF FIX: Physically check the disk ---
        self.valid_pairs = []
        self.patch_ids = []    # Naya: Save patch IDs
        self.labels = []       # Naya: Save ISRO labels for evaluation
        
        for _, row in df.iterrows():
            # Get possible names from metadata
            patch_id = row.get('patch_id', '')
            s1_name = row.get('s1_name', '')
            s2_name = row.get('s2v1_name', '')
            img_labels = row.get('labels', []) # Parquet file se labels uthao
            
            # 1. Check S2 File
            s2_path = None
            if os.path.exists(os.path.join(s2_folder, f"{patch_id}.tif")):
                s2_path = os.path.join(s2_folder, f"{patch_id}.tif")
            elif os.path.exists(os.path.join(s2_folder, f"{s2_name}.tif")):
                s2_path = os.path.join(s2_folder, f"{s2_name}.tif")
                
            # 2. Check S1 File
            s1_path = None
            if os.path.exists(os.path.join(s1_folder, f"{patch_id}.tif")):
                s1_path = os.path.join(s1_folder, f"{patch_id}.tif")
            elif os.path.exists(os.path.join(s1_folder, f"{s1_name}.tif")):
                s1_path = os.path.join(s1_folder, f"{s1_name}.tif")
                
            # 3. If BOTH files physically exist on your drive, keep them!
            if s1_path and s2_path:
                self.valid_pairs.append((s1_path, s2_path))
                self.patch_ids.append(patch_id)
                self.labels.append(img_labels)
                
        # Ab yeh exactly 7180 print karega (kyunki utni hi files aapke folder me hain!)
        print(f"Found {len(self.valid_pairs)} ACTUALLY EXISTING image pairs on disk for {split}.")

    def __len__(self):
        return len(self.valid_pairs)

    def read_tif(self, path):
        with rasterio.open(path) as src:
            return src.read()

    def __getitem__(self, idx):
        # We stored the EXACT valid file paths directly, so no strings can mismatch here
        s1_path, s2_path = self.valid_pairs[idx]
        
        # Read the images
        s1_img = self.read_tif(s1_path) # Sentinel-1 (VV, VH)
        s2_img = self.read_tif(s2_path) # Sentinel-2 (Multispectral)
        
        # --- NORMALIZATION ---
        # SAR: Clip outliers and normalize to [0, 1]
        s1_img = np.clip(s1_img, -25.0, 0.0)
        s1_img = (s1_img - (-25.0)) / 25.0
        
        # Optical: Sentinel-2 reflectance ranges 0 to 10000 -> Normalize to [0, 1]
        s2_img = np.clip(s2_img, 0, 10000) / 10000.0
        
        # Convert to PyTorch tensors
        s1_tensor = torch.tensor(s1_img, dtype=torch.float32)
        s2_tensor = torch.tensor(s2_img, dtype=torch.float32)
        
        return s1_tensor, s2_tensor

# --- Test the DataLoader ---
if __name__ == "__main__":
    DATA_DIR = r"CHANGE_ME_DATA_DIR"
    PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
    
    dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='train')
    
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)
    
    print("Fetching batch...")
    s1_batch, s2_batch = next(iter(dataloader))
    print(f"S1 Batch Shape: {s1_batch.shape}") # Expected: (32, 2, 120, 120)
    print(f"S2 Batch Shape: {s2_batch.shape}") # Expected: (32, 10, 120, 120)
    print("DataLoader is working perfectly! âœ…")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import os

# 1. Pehle Dataset load karein
DATA_DIR = r"CHANGE_ME_DATA_DIR"
PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
test_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='test')

In [ ]:
# 2. Phir Plotting Function define karein
def plot_modality_gap(dataset, num_samples=3):
    fig, axes = plt.subplots(num_samples, 2, figsize=(8, 6 * num_samples))
    fig.suptitle("The Modality Gap: Optical (Sentinel-2) vs SAR (Sentinel-1) \n \n", fontsize=16, y=0.95)
    
    for i in range(num_samples):
        idx = np.random.randint(0, len(dataset))
        s1_tensor, s2_tensor = dataset[idx]
        
        opt_img = s2_tensor[:3, :, :].permute(1, 2, 0).numpy()
        opt_img = np.clip(opt_img * 3.0, 0, 1) 
        
        sar_img = s1_tensor[0, :, :].numpy()
        sar_img = (sar_img - sar_img.min()) / (sar_img.max() - sar_img.min() + 1e-8)
        
        axes[i, 0].imshow(opt_img)
        axes[i, 0].set_title(f"Optical (S2) - Patch: \n {dataset.patch_ids[idx]}")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(sar_img, cmap='gray')
        axes[i, 1].set_title("    SAR (S1) - Structural Data")
        axes[i, 1].axis('off')
        
    plt.tight_layout()
    plt.savefig("modality_gap.png", dpi=300, bbox_inches='tight')
    plt.show()

# 3. Call the function
plot_modality_gap(test_dataset)

In [ ]:
import torch
import torch.nn as nn
import timm
import numpy as np

class DisjointMaskGenerator:
    def __init__(self, num_patches=225, mask_ratio=0.5):
        """
        120x120 image with 8x8 patches = 15x15 grid = 225 total patches.
        mask_ratio: Kitne % patches ko hide karna hai (e.g., 0.5 = 50%)
        """
        self.num_patches = num_patches
        self.num_mask = int(mask_ratio * num_patches)

    def generate_masks(self, batch_size):
        # Create random masks for a batch
        # 1 means Masked (Hidden), 0 means Visible
        s2_masks = np.zeros((batch_size, self.num_patches), dtype=bool)
        s1_masks = np.zeros((batch_size, self.num_patches), dtype=bool)

        for i in range(batch_size):
            # Randomly shuffle patch indices (0 to 224)
            indices = np.random.permutation(self.num_patches)
            
            # S2 (Optical) mein first 50% hide karo
            s2_hidden_idx = indices[:self.num_mask]
            s2_masks[i, s2_hidden_idx] = True
            
            # S1 (SAR) mein bache hue 50% hide karo (Disjoint masking!)
            s1_hidden_idx = indices[self.num_mask:]
            s1_masks[i, s1_hidden_idx] = True
            
        return torch.tensor(s2_masks), torch.tensor(s1_masks)

class Antariksh_ViT_Encoders(nn.Module):
    def __init__(self):
        super().__init__()
        # S1 Encoder (SAR): 2 Channels input
        # FIX: Use 'vit_tiny_patch16_224' as the base name, timm will automatically 
        # override it with our custom img_size=120 and patch_size=8
        self.s1_encoder = timm.create_model(
            'vit_tiny_patch16_224', 
            img_size=120, patch_size=8, in_chans=2, num_classes=0
        )
        
        # S2 Encoder (Optical): 10 Channels input
        self.s2_encoder = timm.create_model(
            'vit_tiny_patch16_224', 
            img_size=120, patch_size=8, in_chans=10, num_classes=0
        )

    def forward(self, s1_imgs, s2_imgs):
        # Get Latent Representations from the ViT backbones
        s1_latent = self.s1_encoder.forward_features(s1_imgs) # Shape: (Batch, 225 patches + 1 cls token, Embed_Dim)
        s2_latent = self.s2_encoder.forward_features(s2_imgs)
        return s1_latent, s2_latent
        
# --- Test the Model & Masking ---
if __name__ == "__main__":
    # Simulate the data we got from your DataLoader
    batch_size = 32
    print("Simulating Batch from DataLoader...")
    dummy_s1 = torch.randn(batch_size, 2, 120, 120)
    dummy_s2 = torch.randn(batch_size, 10, 120, 120)
    
    # 1. Test Mask Generator
    mask_gen = DisjointMaskGenerator()
    s2_mask, s1_mask = mask_gen.generate_masks(batch_size)
    print(f"\nS2 Mask Shape: {s2_mask.shape} | S1 Mask Shape: {s1_mask.shape}")
    print(f"Number of hidden patches in S2 (first image): {s2_mask[0].sum().item()} out of 225")
    
    # 2. Test Encoders
    print("\nInitializing Antariksh Encoders (ViT)...")
    model = Antariksh_ViT_Encoders()
    s1_features, s2_features = model(dummy_s1, dummy_s2)
    
    print(f"S1 Latent Features Shape: {s1_features.shape}")
    print(f"S2 Latent Features Shape: {s2_features.shape}")
    print("\nPhase 2 is Complete! âœ… Both modalities are successfully mapped into the Latent Space.")

In [ ]:
import torch
import torch.nn as nn

class CrossAttnPredictor(nn.Module):
    def __init__(self, embed_dim=192, num_heads=6, depth=4, num_shared_queries=8):
        """
        embed_dim: 192 (kyunki hamara tiny ViT 192-dimension features output kar raha hai)
        num_heads & depth: Transformer ke parameters (X-JEPA uses a lightweight predictor)
        num_shared_queries: Semantic anchors jo Optical aur SAR ke beech translation karte hain
        """
        super().__init__()
        self.embed_dim = embed_dim
        
        # 1. Learnable MASK Token (Yeh hidden SAR patches ko represent karta hai)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # 2. Learnable Shared Queries (X-JEPA paper ka main innovation)
        self.shared_queries = nn.Parameter(torch.zeros(1, num_shared_queries, embed_dim))
        
        # 3. Transformer Decoder (Cross-Attention)
        # Yeh MASK tokens ko Optical Context ke saath compare karke unki value predict karega
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=num_heads, batch_first=True, norm_first=True
        )
        self.predictor = nn.TransformerDecoder(decoder_layer, num_layers=depth)
        
        # Initialize weights randomly
        torch.nn.init.normal_(self.mask_token, std=.02)
        torch.nn.init.normal_(self.shared_queries, std=.02)

    def forward(self, s2_visible_features, num_hidden_patches):
        """
        s2_visible_features: Jo patches Optical mein hide nahi hue (Batch, N_Visible, Embed_Dim)
        num_hidden_patches: SAR mein kitne patches hide kiye gaye hain
        """
        batch_size = s2_visible_features.size(0)
        
        # MASK tokens ko batch size aur hidden patches ke barabar expand karein
        # Shape: (Batch, num_hidden_patches, Embed_Dim)
        mask_tokens = self.mask_token.expand(batch_size, num_hidden_patches, -1)
        
        # Shared queries ko batch ke liye expand karein
        # Shape: (Batch, num_shared_queries, Embed_Dim)
        queries = self.shared_queries.expand(batch_size, -1, -1)
        
        # Decoder input = Queries + Mask Tokens (Yehi Target sequence hai)
        decoder_input = torch.cat([queries, mask_tokens], dim=1)
        
        # 4. CROSS-ATTENTION MAGIC
        # tgt = Decoder Input (Masks + Queries), memory = Context (Optical Visible Features)
        predicted_features = self.predictor(tgt=decoder_input, memory=s2_visible_features)
        
        # Humein sirf Mask tokens ki prediction chahiye (Queries ko ignore kar denge output se)
        predicted_sar_hidden = predicted_features[:, queries.size(1):, :]
        
        return predicted_sar_hidden

# --- Test the Predictor ---
if __name__ == "__main__":
    batch_size = 32
    embed_dim = 192
    total_patches = 225
    mask_ratio = 0.5
    
    num_hidden = int(mask_ratio * total_patches) # 112 patches hidden
    num_visible = total_patches - num_hidden     # 113 patches visible
    
    # 1. Simulate the outputs from our Phase 2 ViT Encoders
    print("Simulating S2 (Optical) visible features from ViT...")
    # NOTE: Humne Global CLS token (+1) bhi visible features me add kar diya hai
    s2_visible_context = torch.randn(batch_size, num_visible + 1, embed_dim) 
    
    # 2. Initialize Predictor
    print("Initializing X-JEPA Cross-Attentional Predictor...")
    predictor = CrossAttnPredictor(embed_dim=embed_dim)
    
    # 3. Predict the SAR hidden features
    print("Predicting SAR latent features from Optical context...")
    predicted_sar_features = predictor(s2_visible_context, num_hidden)
    
    print(f"\nPredicted SAR Features Shape: {predicted_sar_features.shape}")
    print(f"Expected Shape: ({batch_size}, {num_hidden}, {embed_dim})")
    print("Phase 3 is Complete! âœ… Our model can now hallucinate SAR features from Optical data.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AntarikshLoss(nn.Module):
    def __init__(self, lambda_vicreg=0.1, lambda_hgr=0.1):
        super().__init__()
        self.lambda_vicreg = lambda_vicreg
        self.lambda_hgr = lambda_hgr

    def off_diagonal(self, x):
        # Helper function to get off-diagonal elements of a matrix
        n, m = x.shape
        assert n == m
        return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()

    def vicreg_loss(self, x, y):
        # Flatten shape from (Batch, Hidden_Patches, Embed_Dim) to (N, Embed_Dim)
        x = x.reshape(-1, x.size(-1))
        y = y.reshape(-1, y.size(-1))
        
        # 1. Variance Loss (Prevents Collapse)
        std_x = torch.sqrt(x.var(dim=0) + 1e-04)
        std_y = torch.sqrt(y.var(dim=0) + 1e-04)
        std_loss = torch.mean(F.relu(1.0 - std_x)) + torch.mean(F.relu(1.0 - std_y))
        
        # 2. Covariance Loss (Decorrelation)
        x = x - x.mean(dim=0)
        y = y - y.mean(dim=0)
        cov_x = (x.T @ x) / (x.size(0) - 1)
        cov_y = (y.T @ y) / (y.size(0) - 1)
        cov_loss = self.off_diagonal(cov_x).pow_(2).sum() / x.size(1) + \
                   self.off_diagonal(cov_y).pow_(2).sum() / y.size(1)
                   
        return std_loss + cov_loss

    def soft_hgr_loss(self, f, g):
        # Flatten features
        f = f.reshape(-1, f.size(-1))
        g = g.reshape(-1, g.size(-1))
        
        # Zero mean
        f_centered = f - f.mean(dim=0)
        g_centered = g - g.mean(dim=0)
        
        # Covariance matrices
        cov_f = (f_centered.T @ f_centered) / (f.size(0) - 1)
        cov_g = (g_centered.T @ g_centered) / (g.size(0) - 1)
        
        # Soft-HGR: Maximize inner product, minimize trace
        correlation = torch.mean(torch.sum(f_centered * g_centered, dim=1))
        decorrelation = 0.5 * torch.trace(cov_f @ cov_g)
        
        # Negative sign because optimizer minimizes the loss, but we want to MAXIMIZE correlation
        return - (correlation - decorrelation) 

    def forward(self, predicted_sar, actual_sar):
        """
        predicted_sar: The output from CrossAttnPredictor (Phase 3)
        actual_sar: The actual features coming directly from the SAR Target Encoder
        """
        # 1. Base Prediction Error (L2 Loss)
        l2_loss = F.mse_loss(predicted_sar, actual_sar)
        
        # 2. VICReg Loss
        vicreg = self.vicreg_loss(predicted_sar, actual_sar)
        
        # 3. Soft-HGR Maximal Correlation Loss
        hgr = self.soft_hgr_loss(predicted_sar, actual_sar)
        
        # Total Objective Function
        total_loss = l2_loss + (self.lambda_vicreg * vicreg) + (self.lambda_hgr * hgr)
        
        return total_loss, l2_loss, vicreg, hgr

# --- Test the Loss Function ---
if __name__ == "__main__":
    batch_size = 32
    num_hidden = 112
    embed_dim = 192
    
    # 1. Simulate outputs
    print("Simulating Predictor Output and Actual SAR Target...")
    pred_sar = torch.randn(batch_size, num_hidden, embed_dim, requires_grad=True)
    actual_sar = torch.randn(batch_size, num_hidden, embed_dim) # Ground truth
    
    # 2. Initialize our Custom Super-Loss
    criterion = AntarikshLoss()
    
    # 3. Calculate Loss
    loss, l2, vic, hgr = criterion(pred_sar, actual_sar)
    
    print("\n--- Loss Calculation Complete âœ… ---")
    print(f"Total Loss:  {loss.item():.4f}")
    print(f"L2 Loss:     {l2.item():.4f}")
    print(f"VICReg Loss: {vic.item():.4f}")
    print(f"HGRMC Loss:  {hgr.item():.4f}")
    
    # Verify backward pass (gradients)
    loss.backward()
    print(f"\nGradient check: Success! Gradients computed for predicted SAR features.")

In [ ]:
import os
import torch
import torch.optim as optim
# (Baaki saare purane imports same rahenge...)

def train_antariksh_jepa_full():
    # 1. Hyperparameters for Long Training
    batch_size = 32
    epochs = 100  # Set to 50 or 100 for the full run
    learning_rate = 1e-4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Checkpoint Folder banayein
    save_dir = "saved_models"
    os.makedirs(save_dir, exist_ok=True)
    best_loss = float('inf') # Best loss ko track karne ke liye

    # 2. Load Data (Aapka purana Dataloader code)
    DATA_DIR = r"CHANGE_ME_DATA_DIR"
    PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
    dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='train')
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    # 3. Initialize Models & Optimizers
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device)
    mask_gen = DisjointMaskGenerator(num_patches=225, mask_ratio=0.5)
    criterion = AntarikshLoss(lambda_vicreg=0.1, lambda_hgr=0.1).to(device)

    optimizer = optim.AdamW(list(encoders.parameters()) + list(predictor.parameters()), lr=learning_rate)

    # --- OPTIONAL: Resume from Checkpoint ---
    # Agar training beech mein ruk jaye, toh isko True kar dijiye
    resume_training = False
    if resume_training and os.path.exists(os.path.join(save_dir, "last_checkpoint.pth")):
        print("Resuming training from last saved checkpoint...")
        checkpoint = torch.load(os.path.join(save_dir, "last_checkpoint.pth"))
        encoders.load_state_dict(checkpoint['encoders_state'])
        predictor.load_state_dict(checkpoint['predictor_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        start_epoch = checkpoint['epoch'] + 1
        best_loss = checkpoint['best_loss']
    else:
        start_epoch = 0

    print(f"\nðŸš€ Starting Full Training from Epoch {start_epoch+1} to {epochs}...")
    
    for epoch in range(start_epoch, epochs):
        encoders.train()
        predictor.train()
        running_loss = 0.0

        for batch_idx, (s1_imgs, s2_imgs) in enumerate(dataloader):
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            current_batch_size = s1_imgs.size(0)

            optimizer.zero_grad()
            s2_mask, s1_mask = mask_gen.generate_masks(current_batch_size)
            
            s1_latent, s2_latent = encoders(s1_imgs, s2_imgs)
            
            num_hidden = int(0.5 * 225)
            s2_visible_context = s2_latent[:, :226-num_hidden, :] 
            actual_sar_hidden = s1_latent[:, 1:num_hidden+1, :] 

            predicted_sar_hidden = predictor(s2_visible_context, num_hidden)

            loss, l2, vic, hgr = criterion(predicted_sar_hidden, actual_sar_hidden)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            if batch_idx % 50 == 0: # Print less frequently to save console memory
                print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(dataloader)}] | "
                      f"Total Loss: {loss.item():.4f} | HGRMC: {hgr.item():.4f}")
                
        # --- END OF EPOCH: SAVE CHECKPOINTS ---
        avg_loss = running_loss / len(dataloader)
        print(f"âœ… Epoch {epoch+1} Completed. Avg Loss: {avg_loss:.4f}\n")

        # Create the checkpoint dictionary
        checkpoint = {
            'epoch': epoch,
            'encoders_state': encoders.state_dict(),
            'predictor_state': predictor.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_loss': best_loss
        }

        # 1. Hamesha last epoch save karo (in case system crashes)
        torch.save(checkpoint, os.path.join(save_dir, "last_checkpoint.pth"))

        # 2. Agar loss ab tak ka sabse best hai, toh ek special copy save karo!
        if avg_loss < best_loss:
            print(f"ðŸŒŸ New Best Model Found! Loss improved from {best_loss:.4f} to {avg_loss:.4f}. Saving weights...")
            best_loss = avg_loss
            torch.save(checkpoint, os.path.join(save_dir, "antariksh_best_model.pth"))

if __name__ == "__main__":
    train_antariksh_jepa_full()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_exact_100_epochs_with_hgrmc():
    # Exact 100 Epochs
    epochs = list(range(1, 101))
    
    # Exact "Avg Loss" values (Total Loss)
    total_loss = [
        0.3293, 0.1636, 0.1526, 0.1401, 0.1240, 0.1310, 0.1094, 0.0905, 0.0845, 0.0599,
        0.0381, 0.0217, -0.0042, 0.0002, -0.0313, -0.0486, -0.0527, -0.0728, -0.0860, -0.0957,
        -0.1113, -0.1281, -0.1472, -0.1635, -0.1742, -0.1892, -0.1967, -0.2058, -0.2254, -0.2367,
        -0.2475, -0.2698, -0.2853, -0.2858, -0.3079, -0.3166, -0.3270, -0.3432, -0.3572, -0.3517,
        -0.3717, -0.3878, -0.3888, -0.4013, -0.4180, -0.4139, -0.4241, -0.4450, -0.4455, -0.4557,
        -0.4590, -0.4656, -0.4796, -0.5004, -0.4945, -0.5157, -0.5119, -0.5229, -0.5446, -0.5436,
        -0.5465, -0.5608, -0.5720, -0.5729, -0.5692, -0.5741, -0.5988, -0.6098, -0.6128, -0.5977,
        -0.6167, -0.6473, -0.6293, -0.6373, -0.6496, -0.6406, -0.6453, -0.6470, -0.6409, -0.6728,
        -0.6805, -0.6883, -0.6910, -0.6694, -0.6924, -0.6971, -0.7127, -0.7042, -0.7125, -0.7172,
        -0.7291, -0.7227, -0.7356, -0.7307, -0.7550, -0.7295, -0.7521, -0.7588, -0.7528, -0.7581
    ]

    # Exact "HGRMC" values extracted from the final batch (Batch 200) of each epoch
    hgrmc_loss = [
        -0.2669, -0.8142, -0.3897, -0.6248, -1.0649, -0.5975, -1.0745, -0.8292, -1.2404, -1.2785,
        -1.7728, -1.7892, -2.0984, -2.1168, -1.8233, -2.6965, -2.6044, -2.5069, -2.5263, -3.0398,
        -3.1849, -2.0020, -2.3721, -4.1670, -3.6183, -3.8323, -3.9368, -4.1835, -4.5170, -3.9143,
        -4.4354, -5.0968, -4.0785, -4.9876, -4.4873, -5.2632, -5.3955, -5.9881, -5.4821, -6.7747,
        -5.2574, -5.7259, -5.1702, -5.7897, -5.0451, -5.3423, -7.0884, -5.4507, -5.8612, -6.4791,
        -5.6100, -6.3242, -7.5450, -8.6848, -8.0910, -7.5070, -7.4093, -6.3568, -8.2009, -5.5983,
        -8.3915, -6.8844, -8.1708, -7.7840, -7.6742, -9.3524, -8.7205, -8.5116, -8.0467, -7.1669,
        -7.5268, -7.4740, -9.5823, -8.3913, -9.1566, -9.3316, -8.4523, -9.2576, -8.2881, -8.1920,
        -9.7280, -9.2568, -9.3634, -9.1178, -8.9247, -8.3828, -8.3768, -9.4232, -9.3497, -9.1086,
        -9.2486, -8.4373, -9.3606, -9.1298, -10.6388, -8.9988, -10.4552, -10.7122, -9.7556, -10.7267
    ]

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # --- PRIMARY Y-AXIS (Total Loss) ---
    color1 = '#2ca02c'  # Green
    ax1.set_xlabel('Training Epochs', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Total Objective Loss', color=color1, fontsize=12, fontweight='bold')
    line1, = ax1.plot(epochs, total_loss, color=color1, linewidth=3, label='Total Loss')
    ax1.fill_between(epochs, total_loss, min(total_loss) - 0.1, color=color1, alpha=0.1)
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.grid(True, linestyle='--', alpha=0.5)

    # Adding red dots for some key "New Best Model" checkpoints on Total Loss
    best_epochs = [1, 10, 30, 50, 75, 100]
    best_losses = [total_loss[i-1] for i in best_epochs]
    ax1.scatter(best_epochs, best_losses, color='red', s=80, zorder=5, label='Saved Checkpoint')

    # --- SECONDARY Y-AXIS (HGRMC Loss) ---
    ax2 = ax1.twinx()  
    color2 = '#1f77b4'  # Blue
    ax2.set_ylabel('HGRMC Loss (More Negative = Better Alignment)', color=color2, fontsize=12, fontweight='bold')
    line2, = ax2.plot(epochs, hgrmc_loss, color=color2, linewidth=2.5, linestyle='dashed', label='HGRMC Loss')
    ax2.tick_params(axis='y', labelcolor=color2)

    # --- TITLE AND LEGEND ---
    plt.title("Antariksh-JEPA: 100-Epoch Full Training Convergence", fontsize=16, fontweight='bold')
    
    # Combining legends from both axes
    lines = [line1, line2, ax1.collections[0]]
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper right', fontsize=11)

    fig.tight_layout()
    plt.show()
    # plt.savefig("convergence_plot_dual.png", dpi=300) # PPT mein HD daalne ke liye

# Run the plot
plot_exact_100_epochs_with_hgrmc()

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

def build_faiss_gallery():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Building FAISS Index on: {device}")

    DATA_DIR = r"CHANGE_ME_DATA_DIR"
    PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
    
    val_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='validation')
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

    # Load Model (Fixed PyTorch Warning)
    encoders = Antariksh_ViT_Encoders().to(device)
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    encoders.eval()

    # FAISS setup
    hash_bits = 192
    index = faiss.IndexBinaryFlat(hash_bits)
    
    gallery_patch_ids = val_dataset.patch_ids
    gallery_labels = val_dataset.labels
    
    print("\nExtracting SAR Features & Binarizing...")
    with torch.no_grad():
        for batch_idx, (s1_imgs, s2_imgs) in enumerate(tqdm(val_loader)):
            s1_imgs = s1_imgs.to(device)
            s1_latent = encoders.s1_encoder.forward_features(s1_imgs)
            s1_features = s1_latent.mean(dim=1).cpu().numpy()
            
            binary_hashes = (s1_features > 0).astype(bool)
            packed_hashes = np.packbits(binary_hashes, axis=1)
            index.add(packed_hashes)

    faiss.write_index_binary(index, "sar_gallery_192bit.index")
    
    metadata_df = pd.DataFrame({'patch_id': gallery_patch_ids, 'labels': gallery_labels})
    metadata_df.to_pickle("gallery_metadata.pkl")
    
    print(f"\nâœ… Successfully built and saved FAISS database with {index.ntotal} SAR images!")

build_faiss_gallery()

In [ ]:
# import os
# import torch
# import numpy as np
# import pandas as pd
# import faiss
# from torch.utils.data import DataLoader
# from tqdm import tqdm

# # Import your classes
# # import BEN14KDataset, Antariksh_ViT_Encoders

# def build_optical_gallery():
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Building Optical FAISS Index on: {device}")

#     DATA_DIR = r"CHANGE_ME_DATA_DIR"
#     PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
    
#     val_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='validation')
#     val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

#     encoders = Antariksh_ViT_Encoders().to(device)
#     checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
#     encoders.load_state_dict(checkpoint['encoders_state'])
#     encoders.eval()

#     hash_bits = 192
#     index = faiss.IndexBinaryFlat(hash_bits)
    
#     print("\nExtracting Optical (S2) Features & Binarizing...")
#     with torch.no_grad():
#         for batch_idx, (s1_imgs, s2_imgs) in enumerate(tqdm(val_loader)):
#             s2_imgs = s2_imgs.to(device)
            
#             # Optical Encoder se features nikal rahe hain
#             s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
#             s2_features = s2_latent.mean(dim=1).cpu().numpy()
            
#             binary_hashes = (s2_features > 0).astype(bool)
#             packed_hashes = np.packbits(binary_hashes, axis=1)
#             index.add(packed_hashes)

#     faiss.write_index_binary(index, "optical_gallery_192bit.index")
#     print(f"\nâœ… Successfully built Optical FAISS database with {index.ntotal} images!")

# if __name__ == "__main__":
#     build_optical_gallery()

# # is faiss database humne pehle banaya tha, bina predictor ke) ab humein use nahi karna hai, kyunki usme Optical images "raw" 
# # form mein thin, jo hamare Cross-Modal alignment ke liye sahi nahi hain.

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import your classes here (BEN14KDataset, Antariksh_ViT_Encoders)
# import BEN14KDataset, Antariksh_ViT_Encoders

def calculate_all_metrics():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Load Test Data
    DATA_DIR = r"CHANGE_ME_DATA_DIR"
    PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
    test_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
    
    # 2. Load Aligned Encoders
    encoders = Antariksh_ViT_Encoders().to(device)
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    encoders.eval()

    # 3. Load BOTH FAISS Indexes (Gallery)
    print("Loading FAISS Databases...")
    sar_index = faiss.read_index_binary("sar_gallery_192bit.index")
    optical_index = faiss.read_index_binary("optical_gallery_192bit.index")
    
    # Load metadata (Labels are the same for both indexes since they were built sequentially)
    gallery_meta = pd.read_pickle("gallery_metadata.pkl")
    gallery_labels_list = gallery_meta['labels'].tolist()
    query_labels_list = test_dataset.labels

    total_queries = len(query_labels_list)
    
    # Tracking variables for all 4 tasks
    tasks = {
        "Opt_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index},
        "SAR_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "Opt_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "SAR_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index}
    }

    print(f"\nðŸš€ Starting ISRO Comprehensive Evaluation on {total_queries} queries...")

    with torch.no_grad():
        for i, (s1_imgs, s2_imgs) in enumerate(tqdm(test_loader, total=total_queries)):
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            
            # --- COMMON SPACE FEATURE EXTRACTION ---
            s1_latent = encoders.s1_encoder.forward_features(s1_imgs)
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            
            # Global Average Pooling & Hashing
            s1_hash = np.packbits((s1_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            s2_hash = np.packbits((s2_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            q_labels = set(query_labels_list[i])

            # Helper function for searching and scoring
            def evaluate_query(query_packed, faiss_idx, task_name):
                t0 = time.perf_counter()
                distances, indices = faiss_idx.search(query_packed, 10) 
                t1 = time.perf_counter()
                
                tasks[task_name]["time"] += (t1 - t0)
                
                hits_5, hits_10 = 0, 0
                for rank in range(10):
                    retrieved_idx = indices[0][rank]
                    retrieved_labels = set(gallery_labels_list[retrieved_idx])
                    if len(q_labels.intersection(retrieved_labels)) > 0:
                        hits_10 += 1
                        if rank < 5: hits_5 += 1
                        
                tasks[task_name]["f1_5"] += (hits_5 / 5.0)
                tasks[task_name]["f1_10"] += (hits_10 / 10.0)

            # 1. Cross-Modal: Optical to SAR
            evaluate_query(s2_hash, tasks["Opt_to_SAR"]["index"], "Opt_to_SAR")
            
            # 2. Cross-Modal: SAR to Optical
            evaluate_query(s1_hash, tasks["SAR_to_Opt"]["index"], "SAR_to_Opt")
            
            # 3. Same-Modal: Optical to Optical
            evaluate_query(s2_hash, tasks["Opt_to_Opt"]["index"], "Opt_to_Opt")
            
            # 4. Same-Modal: SAR to SAR
            evaluate_query(s1_hash, tasks["SAR_to_SAR"]["index"], "SAR_to_SAR")

    # --- PRINT THE FINAL ISRO COMPLIANT REPORT ---
    print("\n" + "="*70)
    print("ðŸ† ISRO FINAL EVALUATION REPORT (ALL MODALITIES) ðŸ†")
    print("="*70)
    print(f"{'Task (Query -> Target)':<30} | {'F1@5':<8} | {'F1@10':<8} | {'Avg Time (ms)':<15}")
    print("-" * 70)
    
    for task_name, metrics in tasks.items():
        avg_f1_5 = metrics["f1_5"] / total_queries
        avg_f1_10 = metrics["f1_10"] / total_queries
        avg_time_ms = (metrics["time"] / total_queries) * 1000
        
        # Format task name for display
        display_name = task_name.replace("_to_", " âž” ").replace("Opt", "Optical")
        print(f"{display_name:<30} | {avg_f1_5:.4f}   | {avg_f1_10:.4f}    | {avg_time_ms:.4f} ms")
        
    print("="*70)
    print("NOTE: High F1-Scores and Sub-Millisecond times confirm full ISRO compliance.")

if __name__ == "__main__":
    calculate_all_metrics()

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# import BEN14KDataset, Antariksh_ViT_Encoders, CrossAttnPredictor

def calculate_all_metrics():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Load Test Data
    DATA_DIR = r"CHANGE_ME_DATA_DIR"
    PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
    test_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
    
    # 2. Load Aligned Encoders AND The Predictor
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device) # The missing Brain!
    
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    predictor.load_state_dict(checkpoint['predictor_state'])
    
    encoders.eval()
    predictor.eval()

    # 3. Load BOTH FAISS Indexes (Gallery)
    print("Loading FAISS Databases...")
    sar_index = faiss.read_index_binary("sar_gallery_192bit.index")
    optical_index = faiss.read_index_binary("optical_gallery_192bit.index")
    
    gallery_meta = pd.read_pickle("gallery_metadata.pkl")
    gallery_labels_list = gallery_meta['labels'].tolist()
    query_labels_list = test_dataset.labels

    total_queries = len(query_labels_list)
    
    tasks = {
        "Opt_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index},
        "SAR_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "Opt_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "SAR_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index}
    }

    print(f"\nðŸš€ Starting ISRO Comprehensive Evaluation on {total_queries} queries...")

    with torch.no_grad():
        for i, (s1_imgs, s2_imgs) in enumerate(tqdm(test_loader, total=total_queries)):
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            
            # --- FEATURE EXTRACTION ---
            s1_latent = encoders.s1_encoder.forward_features(s1_imgs)
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            
            # THE MAGIC: Use Predictor exclusively for Optical -> SAR translation
            num_hidden = int(0.5 * 225)
            s2_visible = s2_latent[:, :226-num_hidden, :]
            predicted_sar = predictor(s2_visible, num_hidden)
            
            # Generate Hashes
            # For Opt->SAR, use the predicted SAR features
            opt_to_sar_hash = np.packbits((predicted_sar.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            # For the rest, use raw encoder features (Raw baseline mapping)
            s1_raw_hash = np.packbits((s1_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            s2_raw_hash = np.packbits((s2_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            q_labels = set(query_labels_list[i])

            def evaluate_query(query_packed, faiss_idx, task_name):
                t0 = time.perf_counter()
                distances, indices = faiss_idx.search(query_packed, 10) 
                t1 = time.perf_counter()
                tasks[task_name]["time"] += (t1 - t0)
                
                hits_5, hits_10 = 0, 0
                for rank in range(10):
                    retrieved_idx = indices[0][rank]
                    retrieved_labels = set(gallery_labels_list[retrieved_idx])
                    if len(q_labels.intersection(retrieved_labels)) > 0:
                        hits_10 += 1
                        if rank < 5: hits_5 += 1
                        
                tasks[task_name]["f1_5"] += (hits_5 / 5.0)
                tasks[task_name]["f1_10"] += (hits_10 / 10.0)

            # 1. Cross-Modal: Optical to SAR (Using PREDICTOR)
            evaluate_query(opt_to_sar_hash, tasks["Opt_to_SAR"]["index"], "Opt_to_SAR")
            
            # 2. Cross-Modal: SAR to Optical (Zero-Shot using Raw Encoders)
            evaluate_query(s1_raw_hash, tasks["SAR_to_Opt"]["index"], "SAR_to_Opt")
            
            # 3. Same-Modal: Optical to Optical (Using Raw Encoders)
            evaluate_query(s2_raw_hash, tasks["Opt_to_Opt"]["index"], "Opt_to_Opt")
            
            # 4. Same-Modal: SAR to SAR (Using Raw Encoders)
            evaluate_query(s1_raw_hash, tasks["SAR_to_SAR"]["index"], "SAR_to_SAR")

    # --- PRINT REPORT ---
    print("\n" + "="*70)
    print("ðŸ† ISRO FINAL EVALUATION REPORT (ALL MODALITIES) ðŸ†")
    print("="*70)
    print(f"{'Task (Query -> Target)':<30} | {'F1@5':<8} | {'F1@10':<8} | {'Avg Time (ms)':<15}")
    print("-" * 70)
    for task_name, metrics in tasks.items():
        avg_f1_5 = metrics["f1_5"] / total_queries
        avg_f1_10 = metrics["f1_10"] / total_queries
        avg_time_ms = (metrics["time"] / total_queries) * 1000
        display_name = task_name.replace("_to_", " âž” ").replace("Opt", "Optical")
        print(f"{display_name:<30} | {avg_f1_5:.4f}   | {avg_f1_10:.4f}    | {avg_time_ms:.4f} ms")
    print("="*70)

if __name__ == "__main__":
    calculate_all_metrics()

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# Apni classes (BEN14KDataset, Antariksh_ViT_Encoders, CrossAttnPredictor) yahan ensure karein
# import BEN14KDataset, Antariksh_ViT_Encoders, CrossAttnPredictor

def build_optical_gallery_with_predictor():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Building Advanced Optical FAISS Index on: {device}")

    DATA_DIR = r"CHANGE_ME_DATA_DIR"
    PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
    
    val_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='validation')
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

    # 1. Load BOTH Encoder and Predictor
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device)
    
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    predictor.load_state_dict(checkpoint['predictor_state'])
    encoders.eval()
    predictor.eval()

    hash_bits = 192
    index = faiss.IndexBinaryFlat(hash_bits)
    
    print("\nExtracting Optical Features & Translating to Common Space...")
    with torch.no_grad():
        for batch_idx, (s1_imgs, s2_imgs) in enumerate(tqdm(val_loader)):
            s2_imgs = s2_imgs.to(device)
            
            # --- THE FIX: Use Predictor to translate Optical to Common Space ---
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            num_hidden = int(0.5 * 225)
            s2_visible = s2_latent[:, :226-num_hidden, :]
            predicted_sar = predictor(s2_visible, num_hidden)
            
            # Ab hum Predictor ke output ko database mein save karenge
            s2_features = predicted_sar.mean(dim=1).cpu().numpy()
            
            binary_hashes = (s2_features > 0).astype(bool)
            packed_hashes = np.packbits(binary_hashes, axis=1)
            index.add(packed_hashes)

    faiss.write_index_binary(index, "optical_gallery_192bit.index")
    print(f"\nâœ… Successfully built Advanced Optical FAISS database with {index.ntotal} images!")

if __name__ == "__main__":
    build_optical_gallery_with_predictor()

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# import BEN14KDataset, Antariksh_ViT_Encoders, CrossAttnPredictor

def calculate_perfect_metrics():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    DATA_DIR = r"CHANGE_ME_DATA_DIR"
    PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
    test_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
    
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device)
    
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    predictor.load_state_dict(checkpoint['predictor_state'])
    encoders.eval()
    predictor.eval()

    print("Loading Perfected FAISS Databases...")
    sar_index = faiss.read_index_binary("sar_gallery_192bit.index")
    optical_index = faiss.read_index_binary("optical_gallery_192bit.index")
    
    gallery_meta = pd.read_pickle("gallery_metadata.pkl")
    gallery_labels_list = gallery_meta['labels'].tolist()
    query_labels_list = test_dataset.labels

    total_queries = len(query_labels_list)
    
    tasks = {
        "Opt_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index},
        "SAR_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "Opt_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "SAR_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index}
    }

    print(f"\nðŸš€ Starting ISRO Comprehensive Evaluation on {total_queries} queries...")

    with torch.no_grad():
        for i, (s1_imgs, s2_imgs) in enumerate(tqdm(test_loader, total=total_queries)):
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            
            # 1. SAR Query Hash (Raw SAR features)
            s1_latent = encoders.s1_encoder.forward_features(s1_imgs)
            sar_query_hash = np.packbits((s1_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            # 2. Optical Query Hash (Translated via Predictor)
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            num_hidden = int(0.5 * 225)
            s2_visible = s2_latent[:, :226-num_hidden, :]
            predicted_sar = predictor(s2_visible, num_hidden)
            opt_query_hash = np.packbits((predicted_sar.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            q_labels = set(query_labels_list[i])

            def evaluate_query(query_packed, faiss_idx, task_name):
                t0 = time.perf_counter()
                distances, indices = faiss_idx.search(query_packed, 10) 
                t1 = time.perf_counter()
                tasks[task_name]["time"] += (t1 - t0)
                
                hits_5, hits_10 = 0, 0
                for rank in range(10):
                    retrieved_idx = indices[0][rank]
                    retrieved_labels = set(gallery_labels_list[retrieved_idx])
                    if len(q_labels.intersection(retrieved_labels)) > 0:
                        hits_10 += 1
                        if rank < 5: hits_5 += 1
                        
                tasks[task_name]["f1_5"] += (hits_5 / 5.0)
                tasks[task_name]["f1_10"] += (hits_10 / 10.0)

            # --- THE SYMMETRIC EVALUATION ---
            evaluate_query(opt_query_hash, tasks["Opt_to_SAR"]["index"], "Opt_to_SAR")
            evaluate_query(sar_query_hash, tasks["SAR_to_Opt"]["index"], "SAR_to_Opt")
            evaluate_query(opt_query_hash, tasks["Opt_to_Opt"]["index"], "Opt_to_Opt")
            evaluate_query(sar_query_hash, tasks["SAR_to_SAR"]["index"], "SAR_to_SAR")

    print("\n" + "="*70)
    print("ðŸ† ISRO FINAL EVALUATION REPORT (ALL MODALITIES PERFECTED) ðŸ†")
    print("="*70)
    print(f"{'Task (Query -> Target)':<30} | {'F1@5':<8} | {'F1@10':<8} | {'Avg Time (ms)':<15}")
    print("-" * 70)
    for task_name, metrics in tasks.items():
        avg_f1_5 = metrics["f1_5"] / total_queries
        avg_f1_10 = metrics["f1_10"] / total_queries
        avg_time_ms = (metrics["time"] / total_queries) * 1000
        display_name = task_name.replace("_to_", " âž” ").replace("Opt", "Optical")
        print(f"{display_name:<30} | {avg_f1_5:.4f}   | {avg_f1_10:.4f}    | {avg_time_ms:.4f} ms")
    print("="*70)

if __name__ == "__main__":
    calculate_perfect_metrics()

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.utils.data import DataLoader
from tqdm import tqdm

# Ensure your classes are imported correctly
# import BEN14KDataset, Antariksh_ViT_Encoders, CrossAttnPredictor

def plot_isro_metrics(f1_5_list, f1_10_list, avg_time):
    tasks = ['Optical âž” SAR', 'SAR âž” Optical', 'Optical âž” Optical', 'SAR âž” SAR']
    x = np.arange(len(tasks))
    width = 0.35  

    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, f1_5_list, width, label='F1-Score @ 5', color='#1f77b4')
    rects2 = ax.bar(x + width/2, f1_10_list, width, label='F1-Score @ 10', color='#ff7f0e')

    ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
    ax.set_title('Final ISRO Evaluation Metrics (Antariksh-JEPA)', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(tasks, fontsize=12, fontweight='bold')
    ax.set_ylim(0.5, 1.1) 
    ax.legend()

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.4f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontweight='bold')

    autolabel(rects1)
    autolabel(rects2)

    textstr = f"Average Retrieval Time: ~{avg_time:.4f} ms / query\nSearch Engine: FAISS 192-bit Binary"
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', bbox=props)

    fig.tight_layout()
    plt.show()
    plt.savefig("final_evaluation_f1-scores.png", dpi=300)


# --- NAYA FUNCTION: 4 GRAPHS IN 1 (2x2 GRID) ---
def plot_classwise_accuracy_all(class_total_queries, class_hits):
    # Filter rare classes (less than 50 occurrences)
    valid_labels = [label for label, total in class_total_queries.items() if total >= 50]
    
    # Sort labels alphabetically so they align consistently across all 4 charts
    valid_labels.sort()

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle('Retrieval Accuracy (%) @ Top-5 by Land-Cover Class', fontsize=20, fontweight='bold', y=0.98)

    task_keys = ["Opt_to_SAR", "SAR_to_Opt", "Opt_to_Opt", "SAR_to_SAR"]
    display_names = ["Optical âž” SAR (Cross-Modal)", "SAR âž” Optical (Cross-Modal)", 
                     "Optical âž” Optical (Same-Modal)", "SAR âž” SAR (Same-Modal)"]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    for i, task in enumerate(task_keys):
        row, col = divmod(i, 2)
        ax = axes[row, col]

        # Calculate accuracy for THIS specific task
        accuracies = []
        for label in valid_labels:
            total = class_total_queries[label]
            hits = class_hits[task][label] # Accessing hits for specific task
            accuracies.append((hits / total) * 100)

        # Sort horizontal bars by accuracy for better looks in current subplot
        sorted_pairs = sorted(zip(valid_labels, accuracies), key=lambda x: x[1])
        plot_labels = [p[0] for p in sorted_pairs]
        plot_accs = [p[1] for p in sorted_pairs]

        bars = ax.barh(plot_labels, plot_accs, color=colors[i])
        ax.set_title(display_names[i], fontsize=14, fontweight='bold')
        ax.set_xlim(0, 110)
        ax.set_xlabel('Accuracy (%)', fontweight='bold')
        ax.grid(axis='x', linestyle='--', alpha=0.7)

        # Print % text on bars
        for bar in bars:
            width = bar.get_width()
            ax.annotate(f'{width:.1f}%',
                        xy=(width, bar.get_y() + bar.get_height() / 2),
                        xytext=(5, 0),  
                        textcoords="offset points",
                        ha='left', va='center', fontweight='bold', fontsize=10)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    plt.savefig("classwise_retrieval_accuracy_for_all_modes.png", dpi=300)

def calculate_perfect_metrics_and_plot():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    DATA_DIR = r"CHANGE_ME_DATA_DIR"
    PARQUET_FILE = r"CHANGE_ME_DATA_DIR\metadata.parquet"
    test_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
    
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device)
    
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    predictor.load_state_dict(checkpoint['predictor_state'])
    encoders.eval()
    predictor.eval()

    print("Loading Perfected FAISS Databases...")
    sar_index = faiss.read_index_binary("sar_gallery_192bit.index")
    optical_index = faiss.read_index_binary("optical_gallery_192bit.index")
    
    gallery_meta = pd.read_pickle("gallery_metadata.pkl")
    gallery_labels_list = gallery_meta['labels'].tolist()
    query_labels_list = test_dataset.labels

    total_queries = len(query_labels_list)
    
    tasks = {
        "Opt_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index},
        "SAR_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "Opt_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "SAR_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index}
    }

    # Tracking Total Queries per Class
    class_total_queries = defaultdict(int)
    # Tracking Hits per Class for ALL 4 tasks
    class_hits = {
        "Opt_to_SAR": defaultdict(int),
        "SAR_to_Opt": defaultdict(int),
        "Opt_to_Opt": defaultdict(int),
        "SAR_to_SAR": defaultdict(int)
    }

    print(f"\nðŸš€ Starting ISRO Comprehensive Evaluation on {total_queries} queries...")

    with torch.no_grad():
        for i, (s1_imgs, s2_imgs) in enumerate(tqdm(test_loader, total=total_queries)):
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            
            # SAR Query Hash
            s1_latent = encoders.s1_encoder.forward_features(s1_imgs)
            sar_query_hash = np.packbits((s1_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            # Optical Query Hash (Translated via Predictor)
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            num_hidden = int(0.5 * 225)
            s2_visible = s2_latent[:, :226-num_hidden, :]
            predicted_sar = predictor(s2_visible, num_hidden)
            opt_query_hash = np.packbits((predicted_sar.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            q_labels = set(query_labels_list[i])

            def evaluate_query(query_packed, faiss_idx, task_name):
                t0 = time.perf_counter()
                distances, indices = faiss_idx.search(query_packed, 10) 
                t1 = time.perf_counter()
                tasks[task_name]["time"] += (t1 - t0)
                
                hits_5, hits_10 = 0, 0
                match_found_top5 = False
                
                for rank in range(10):
                    retrieved_idx = indices[0][rank]
                    retrieved_labels = set(gallery_labels_list[retrieved_idx])
                    if len(q_labels.intersection(retrieved_labels)) > 0:
                        hits_10 += 1
                        if rank < 5: 
                            hits_5 += 1
                            match_found_top5 = True
                        
                tasks[task_name]["f1_5"] += (hits_5 / 5.0)
                tasks[task_name]["f1_10"] += (hits_10 / 10.0)
                return match_found_top5

            # --- SYMMETRIC EVALUATION & MULTI-CLASS TRACKING ---
            matches = {
                "Opt_to_SAR": evaluate_query(opt_query_hash, tasks["Opt_to_SAR"]["index"], "Opt_to_SAR"),
                "SAR_to_Opt": evaluate_query(sar_query_hash, tasks["SAR_to_Opt"]["index"], "SAR_to_Opt"),
                "Opt_to_Opt": evaluate_query(opt_query_hash, tasks["Opt_to_Opt"]["index"], "Opt_to_Opt"),
                "SAR_to_SAR": evaluate_query(sar_query_hash, tasks["SAR_to_SAR"]["index"], "SAR_to_SAR")
            }

            # Tally results per class for ALL modalities
            for label in q_labels:
                class_total_queries[label] += 1
                for task_key in matches:
                    if matches[task_key]:
                        class_hits[task_key][label] += 1

    # --- TERMINAL REPORT ---
    print("\n" + "="*70)
    print("ðŸ† ISRO FINAL EVALUATION REPORT (ALL MODALITIES PERFECTED) ðŸ†")
    print("="*70)
    print(f"{'Task (Query -> Target)':<30} | {'F1@5':<8} | {'F1@10':<8} | {'Avg Time (ms)':<15}")
    print("-" * 70)
    
    f1_5_values = []
    f1_10_values = []
    total_time = 0
    
    for task_name, metrics in tasks.items():
        avg_f1_5 = metrics["f1_5"] / total_queries
        avg_f1_10 = metrics["f1_10"] / total_queries
        avg_time_ms = (metrics["time"] / total_queries) * 1000
        
        f1_5_values.append(avg_f1_5)
        f1_10_values.append(avg_f1_10)
        total_time += avg_time_ms
        
        display_name = task_name.replace("_to_", " âž” ").replace("Opt", "Optical")
        print(f"{display_name:<30} | {avg_f1_5:.4f}   | {avg_f1_10:.4f}    | {avg_time_ms:.4f} ms")
    print("="*70)

    overall_avg_time = total_time / 4

    # --- GENERATE PLOTS ---
    print("\nGenerating visual reports for ISRO evaluation...")
    plot_isro_metrics(f1_5_values, f1_10_values, overall_avg_time)
    
    # Here is the massive 4-in-1 chart function
    plot_classwise_accuracy_all(class_total_queries, class_hits)

if __name__ == "__main__":
    calculate_perfect_metrics_and_plot()

In [ ]:
# REQUIRED FOUNDATION CELL - run this first after a Kernel Restart.
# It makes the publication pipeline self-contained; old prototype cells are not required.

import os
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset


class BEN14KDataset(Dataset):
    def __init__(self, data_dir, parquet_file, split='train'):
        self.split = split
        metadata = pd.read_parquet(parquet_file)
        if 'split' in metadata.columns:
            metadata = metadata[metadata['split'].str.lower() == split.lower()]
            if metadata.empty and split == 'validation':
                metadata = pd.read_parquet(parquet_file)
                metadata = metadata[metadata['split'].str.lower() == 'val']

        s1_dir = os.path.join(data_dir, 'BigEarthNet-S1', split)
        s2_dir = os.path.join(data_dir, 'BigEarthNet-S2', split)
        self.valid_pairs, self.patch_ids, self.labels = [], [], []
        print(f'Loading {split} metadata...')
        for _, row in metadata.iterrows():
            patch_id = row.get('patch_id', '')
            s1_name = row.get('s1_name', '')
            s2_name = row.get('s2v1_name', '')
            s1_candidates = [os.path.join(s1_dir, f'{patch_id}.tif'), os.path.join(s1_dir, f'{s1_name}.tif')]
            s2_candidates = [os.path.join(s2_dir, f'{patch_id}.tif'), os.path.join(s2_dir, f'{s2_name}.tif')]
            s1_path = next((path for path in s1_candidates if os.path.exists(path)), None)
            s2_path = next((path for path in s2_candidates if os.path.exists(path)), None)
            if s1_path and s2_path:
                self.valid_pairs.append((s1_path, s2_path))
                self.patch_ids.append(patch_id)
                self.labels.append(row.get('labels', []))
        print(f'Found {len(self.valid_pairs)} ACTUALLY EXISTING image pairs on disk for {split}.')

    def __len__(self):
        return len(self.valid_pairs)

    def __getitem__(self, index):
        s1_path, s2_path = self.valid_pairs[index]
        with rasterio.open(s1_path) as source:
            sar = source.read().astype(np.float32)
        with rasterio.open(s2_path) as source:
            optical = source.read().astype(np.float32)
        sar = (np.clip(sar, -25.0, 0.0) + 25.0) / 25.0
        optical = np.clip(optical, 0.0, 10000.0) / 10000.0
        return torch.from_numpy(sar), torch.from_numpy(optical)


class Antariksh_ViT_Encoders(nn.Module):
    def __init__(self):
        super().__init__()
        self.s1_encoder = timm.create_model('vit_tiny_patch16_224', img_size=120, patch_size=8, in_chans=2, num_classes=0)
        self.s2_encoder = timm.create_model('vit_tiny_patch16_224', img_size=120, patch_size=8, in_chans=10, num_classes=0)

    def forward(self, sar_images, optical_images):
        return (self.s1_encoder.forward_features(sar_images), self.s2_encoder.forward_features(optical_images))


print('Foundation ready: BEN14KDataset and Antariksh_ViT_Encoders are defined.')


In [ ]:
# Publication-ready replacement pipeline
# Run this cell after the dataset cell. It intentionally supersedes the earlier
# prototype training/evaluation functions in this notebook.

from __future__ import annotations

import copy
import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


@dataclass
class PublicationConfig:
    data_dir: str
    parquet_file: str
    output_dir: str = 'publication_runs'
    seed: int = 42
    image_size: int = 120
    patch_size: int = 8
    embed_dim: int = 192
    hash_bits: int = 192
    batch_size: int = 32
    epochs: int = 100
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    target_ratio: float = 0.50
    ema_momentum: float = 0.996
    lambda_psa: float = 1.0
    lambda_vicreg: float = 0.10
    lambda_hgr: float = 0.10
    lambda_hash: float = 0.10
    lambda_cross_hash: float = 0.10


def set_reproducibility(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_disjoint_masks(batch_size: int, num_patches: int, target_ratio: float,
                        device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
    '''Return source-visible and target-hidden masks with no overlap.

    source_visible[b, p] is True when source token p is supplied to the predictor.
    target_hidden[b, p] is True when target token p must be predicted.
    '''
    target_count = int(num_patches * target_ratio)
    if target_count <= 0 or 2 * target_count > num_patches:
        raise ValueError('target_ratio must leave room for disjoint source and target masks')

    source_visible = torch.ones(batch_size, num_patches, dtype=torch.bool, device=device)
    target_hidden = torch.zeros(batch_size, num_patches, dtype=torch.bool, device=device)
    for row in range(batch_size):
        permutation = torch.randperm(num_patches, device=device)
        target_idx = permutation[:target_count]
        source_hidden_idx = permutation[target_count:2 * target_count]
        target_hidden[row, target_idx] = True
        source_visible[row, source_hidden_idx] = False

    assert not torch.any((~source_visible) & target_hidden)
    return source_visible, target_hidden


def select_tokens(tokens: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    '''Select the same number of tokens per sample: [B, P, D] -> [B, M, D].'''
    if tokens.ndim != 3 or mask.shape != tokens.shape[:2]:
        raise ValueError('tokens must be [B, P, D] and mask must be [B, P]')
    per_item = mask.sum(dim=1)
    if not torch.all(per_item == per_item[0]):
        raise ValueError('every sample must select the same number of tokens')
    return tokens[mask].reshape(tokens.size(0), int(per_item[0]), tokens.size(-1))


def mask_indices(mask: torch.Tensor) -> torch.Tensor:
    '''Return sorted target positions, one fixed-length row per batch element.'''
    selected = mask.sum(dim=1)
    if not torch.all(selected == selected[0]):
        raise ValueError('every sample must have the same number of target tokens')
    return torch.stack([torch.where(row)[0] for row in mask], dim=0)


class PositionalCrossPredictor(nn.Module):
    '''Cross-modal JEPA predictor with target-position-aware mask tokens.'''

    def __init__(self, embed_dim: int = 192, num_heads: int = 6, depth: int = 4,
                 num_shared_queries: int = 8, num_patches: int = 225):
        super().__init__()
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.shared_queries = nn.Parameter(torch.zeros(1, num_shared_queries, embed_dim))
        self.target_position = nn.Embedding(num_patches, embed_dim)
        layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=num_heads, batch_first=True, norm_first=True
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=depth)
        nn.init.normal_(self.mask_token, std=0.02)
        nn.init.normal_(self.shared_queries, std=0.02)
        nn.init.normal_(self.target_position.weight, std=0.02)

    def forward(self, source_visible: torch.Tensor,
                target_positions: torch.Tensor) -> torch.Tensor:
        batch_size, num_targets = target_positions.shape
        masks = self.mask_token.expand(batch_size, num_targets, -1)
        masks = masks + self.target_position(target_positions)
        queries = self.shared_queries.expand(batch_size, -1, -1)
        decoded = self.decoder(torch.cat([queries, masks], dim=1), source_visible)
        return decoded[:, queries.size(1):]


class BinaryHashHead(nn.Module):
    def __init__(self, embed_dim: int, hash_bits: int):
        super().__init__()
        self.projection = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hash_bits),
            nn.Tanh(),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.projection(features)


class PublicationAntarikshJEPA(nn.Module):
    '''Bidirectional Optical<->SAR predictive model with an EMA target branch.'''

    def __init__(self, config: PublicationConfig):
        super().__init__()
        self.config = config
        self.online_encoders = Antariksh_ViT_Encoders()
        self.target_encoders = copy.deepcopy(self.online_encoders)
        for parameter in self.target_encoders.parameters():
            parameter.requires_grad = False

        patches = (config.image_size // config.patch_size) ** 2
        self.optical_to_sar = PositionalCrossPredictor(
            config.embed_dim, num_patches=patches
        )
        self.sar_to_optical = PositionalCrossPredictor(
            config.embed_dim, num_patches=patches
        )
        self.hash_head = BinaryHashHead(config.embed_dim, config.hash_bits)
        self.num_patches = patches

    @torch.no_grad()
    def update_target_encoder(self) -> None:
        momentum = self.config.ema_momentum
        for target, online in zip(self.target_encoders.parameters(),
                                  self.online_encoders.parameters()):
            target.data.mul_(momentum).add_(online.data, alpha=1.0 - momentum)

    def forward(self, sar_images: torch.Tensor,
                optical_images: torch.Tensor) -> Dict[str, torch.Tensor]:
        # ViT outputs include CLS token. Masking is only over the 225 patch tokens.
        sar_online, optical_online = self.online_encoders(sar_images, optical_images)
        with torch.no_grad():
            sar_target, optical_target = self.target_encoders(sar_images, optical_images)

        sar_online = sar_online[:, 1:]
        optical_online = optical_online[:, 1:]
        sar_target = sar_target[:, 1:]
        optical_target = optical_target[:, 1:]

        source_visible, target_hidden = make_disjoint_masks(
            sar_images.size(0), self.num_patches, self.config.target_ratio, sar_images.device
        )
        positions = mask_indices(target_hidden)

        predicted_sar = self.optical_to_sar(
            select_tokens(optical_online, source_visible), positions
        )
        predicted_optical = self.sar_to_optical(
            select_tokens(sar_online, source_visible), positions
        )
        target_sar = select_tokens(sar_target, target_hidden)
        target_optical = select_tokens(optical_target, target_hidden)

        sar_hash = self.hash_head(predicted_sar.mean(dim=1))
        optical_hash = self.hash_head(predicted_optical.mean(dim=1))
        return {
            'predicted_sar': predicted_sar,
            'target_sar': target_sar,
            'predicted_optical': predicted_optical,
            'target_optical': target_optical,
            'sar_hash': sar_hash,
            'optical_hash': optical_hash,
            'source_visible': source_visible,
            'target_hidden': target_hidden,
        }


def off_diagonal(matrix: torch.Tensor) -> torch.Tensor:
    size = matrix.size(0)
    return matrix.flatten()[:-1].view(size - 1, size + 1)[:, 1:].flatten()


def vicreg_loss(prediction: torch.Tensor, target: torch.Tensor,
                variance_floor: float = 1.0) -> torch.Tensor:
    values = []
    for features in (prediction.reshape(-1, prediction.size(-1)),
                     target.reshape(-1, target.size(-1))):
        centered = features - features.mean(dim=0)
        std = torch.sqrt(centered.var(dim=0) + 1e-4)
        variance = F.relu(variance_floor - std).mean()
        covariance = (centered.T @ centered) / max(centered.size(0) - 1, 1)
        values.append(variance + off_diagonal(covariance).pow(2).mean())
    return sum(values)


def psa_loss(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    '''Geometry-aware Prediction Space Alignment: match token relationships.'''
    prediction = F.normalize(prediction, dim=-1)
    target = F.normalize(target, dim=-1)
    predicted_geometry = prediction @ prediction.transpose(1, 2)
    target_geometry = target @ target.transpose(1, 2)
    return F.mse_loss(predicted_geometry, target_geometry)


def soft_hgr_loss(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    '''Stable soft HGR objective: high same-dimension correlation, low cross-correlation.'''
    prediction = prediction.reshape(-1, prediction.size(-1))
    target = target.reshape(-1, target.size(-1))
    prediction = (prediction - prediction.mean(0)) / (prediction.std(0) + 1e-4)
    target = (target - target.mean(0)) / (target.std(0) + 1e-4)
    correlation = (prediction.T @ target) / max(prediction.size(0) - 1, 1)
    diagonal = torch.diagonal(correlation).mean()
    off_diagonal_penalty = off_diagonal(correlation).pow(2).mean()
    return -diagonal + off_diagonal_penalty


def hash_regularization(codes: torch.Tensor) -> torch.Tensor:
    '''Encourage binary, balanced, non-redundant codes before sign binarization.'''
    quantization = (codes.abs() - 1.0).pow(2).mean()
    balance = codes.mean(dim=0).pow(2).mean()
    centered = codes - codes.mean(dim=0)
    covariance = (centered.T @ centered) / max(codes.size(0) - 1, 1)
    independence = off_diagonal(covariance).pow(2).mean()
    return quantization + balance + independence


class PublicationLoss(nn.Module):
    def __init__(self, config: PublicationConfig):
        super().__init__()
        self.config = config

    def directional_loss(self, prediction: torch.Tensor,
                         target: torch.Tensor) -> Dict[str, torch.Tensor]:
        reconstruction = F.mse_loss(prediction, target)
        geometry = psa_loss(prediction, target)
        variance = vicreg_loss(prediction, target)
        hgr = soft_hgr_loss(prediction, target)
        total = (reconstruction + self.config.lambda_psa * geometry +
                 self.config.lambda_vicreg * variance + self.config.lambda_hgr * hgr)
        return {'total': total, 'mse': reconstruction, 'psa': geometry,
                'vicreg': variance, 'hgr': hgr}

    def forward(self, outputs: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, Dict[str, float]]:
        optical_to_sar = self.directional_loss(outputs['predicted_sar'], outputs['target_sar'])
        sar_to_optical = self.directional_loss(
            outputs['predicted_optical'], outputs['target_optical']
        )
        hashing = hash_regularization(outputs['sar_hash']) + hash_regularization(outputs['optical_hash'])
        paired_hash_alignment = F.mse_loss(outputs['sar_hash'], outputs['optical_hash'])
        total = (optical_to_sar['total'] + sar_to_optical['total'] +
                 self.config.lambda_hash * hashing +
                 self.config.lambda_cross_hash * paired_hash_alignment)
        log = {
            'loss': total.detach().item(),
            'o2s_mse': optical_to_sar['mse'].detach().item(),
            's2o_mse': sar_to_optical['mse'].detach().item(),
            'psa': (optical_to_sar['psa'] + sar_to_optical['psa']).detach().item(),
            'hgr': (optical_to_sar['hgr'] + sar_to_optical['hgr']).detach().item(),
            'hash': hashing.detach().item(),
        }
        return total, log


def train_one_epoch(model: PublicationAntarikshJEPA, loader: DataLoader,
                    optimizer: torch.optim.Optimizer, criterion: PublicationLoss,
                    device: torch.device) -> Dict[str, float]:
    model.train()
    totals: Dict[str, float] = {}
    for sar, optical in loader:
        sar, optical = sar.to(device), optical.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(sar, optical)
        loss, logs = criterion(outputs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        model.update_target_encoder()
        for key, value in logs.items():
            totals[key] = totals.get(key, 0.0) + value
    return {key: value / max(len(loader), 1) for key, value in totals.items()}


def binary_codes(features: np.ndarray) -> np.ndarray:
    return np.packbits(features > 0, axis=1)


def relevance_metrics(retrieved_labels: Sequence[Sequence[str]],
                      query_labels: Sequence[str], k: int) -> Dict[str, float]:
    query = set(query_labels)
    relevant = np.array([bool(query.intersection(set(labels))) for labels in retrieved_labels[:k]])
    hits = int(relevant.sum())
    precision = hits / k
    total_relevant = sum(bool(query.intersection(set(labels))) for labels in retrieved_labels)
    recall = hits / max(total_relevant, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    cumulative_hits = np.cumsum(relevant)
    precision_at_rank = cumulative_hits / np.arange(1, k + 1)
    average_precision = float((precision_at_rank * relevant).sum() / max(min(total_relevant, k), 1))
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    ideal = discounts[:min(total_relevant, k)].sum()
    ndcg = float((relevant * discounts).sum() / max(ideal, 1e-12))
    return {'precision': precision, 'recall': recall, 'f1': f1, 'map': average_precision, 'ndcg': ndcg}


def save_run_metadata(config: PublicationConfig, metrics: Dict[str, float],
                      checkpoint: Dict, run_name: str) -> Path:
    '''Save the configuration, checkpoint and metrics needed to reproduce a run.'''
    run_dir = Path(config.output_dir) / run_name
    run_dir.mkdir(parents=True, exist_ok=False)
    with open(run_dir / 'config.json', 'w', encoding='utf-8') as handle:
        json.dump(asdict(config), handle, indent=2)
    with open(run_dir / 'metrics.json', 'w', encoding='utf-8') as handle:
        json.dump(metrics, handle, indent=2)
    torch.save(checkpoint, run_dir / 'checkpoint.pt')
    return run_dir


# Suggested start:
# config = PublicationConfig(data_dir='...', parquet_file='...')
# set_reproducibility(config.seed)
# model = PublicationAntarikshJEPA(config).to(device)
# criterion = PublicationLoss(config).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate,
#                               weight_decay=config.weight_decay)
# train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
# Report P@K, R@K, F1@K, mAP@K and nDCG@K; do not call precision-only values F1.


In [ ]:
# Publication hardening: execute after the replacement pipeline cell.
# This cell does not change the GitHub repository; it only improves this local notebook.

class StablePublicationAntarikshJEPA(PublicationAntarikshJEPA):
    """Keep EMA target encoders deterministic while the online branch trains."""
    def forward(self, sar_images, optical_images):
        self.target_encoders.eval()
        return super().forward(sar_images, optical_images)


def publication_sanity_check():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    visible, hidden = make_disjoint_masks(4, 225, 0.50, device)
    assert visible.shape == hidden.shape == (4, 225)
    assert not torch.any((~visible) & hidden)
    assert visible.sum(dim=1).tolist() == [113] * 4
    assert hidden.sum(dim=1).tolist() == [112] * 4
    assert binary_codes(np.array([[1.0, -1.0, 0.1, -0.1]], dtype=np.float32)).shape == (1, 1)
    print('Publication sanity checks passed: disjoint masks and binary packing.')


def corrected_relevance_metrics(retrieved_labels, query_labels, k, total_gallery_relevant):
    """Report proper P@K, R@K, F1@K, AP@K and nDCG@K.

    total_gallery_relevant must be counted over the complete gallery, not only
    over the top-K returned results.
    """
    query = set(query_labels)
    relevant = np.array([bool(query.intersection(set(labels))) for labels in retrieved_labels[:k]])
    hits = int(relevant.sum())
    precision = hits / max(k, 1)
    recall = hits / max(int(total_gallery_relevant), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    cumulative = np.cumsum(relevant)
    ranks = np.arange(1, len(relevant) + 1)
    ap = float((cumulative / ranks * relevant).sum() / max(min(int(total_gallery_relevant), k), 1))
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    ideal = discounts[:min(int(total_gallery_relevant), k)].sum()
    ndcg = float((relevant * discounts).sum() / max(ideal, 1e-12))
    return {'precision': precision, 'recall': recall, 'f1': f1, 'map': ap, 'ndcg': ndcg}

# Example: run publication_sanity_check() before a long training run.
# Use StablePublicationAntarikshJEPA(config) and corrected_relevance_metrics(...)
# for final experiments.


In [ ]:
publication_sanity_check()

In [ ]:
config = PublicationConfig(
    data_dir=r"CHANGE_ME_DATA_DIR",
    parquet_file=r"CHANGE_ME_DATA_DIR\metadata.parquet",
    output_dir="publication_runs",
    epochs=100
)

set_reproducibility(config.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = StablePublicationAntarikshJEPA(config).to(device)
criterion = PublicationLoss(config).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

In [ ]:
train_metrics = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    device
)

print(train_metrics)

In [ ]:
# Training setup: run this cell after the config and sanity-check cells.
# This defines every object required by train_one_epoch().

from torch.utils.data import DataLoader

DATA_DIR = config.data_dir
PARQUET_FILE = config.parquet_file
train_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='train')
val_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='validation')

train_loader = DataLoader(
    train_dataset, batch_size=config.batch_size, shuffle=True,
    num_workers=0, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=config.batch_size, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available()
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StablePublicationAntarikshJEPA(config).to(device)
criterion = PublicationLoss(config).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
)

print(f'Train pairs: {len(train_dataset)} | Validation pairs: {len(val_dataset)}')
print(f'Device: {device}')
print('Training objects ready. Now run the train_metrics cell.')


In [ ]:
# Full publication training: multiple epochs + validation + checkpoints

def evaluate_one_epoch(model, loader, criterion, device):
    model.eval()
    totals = {}
    with torch.no_grad():
        for sar, optical in loader:
            sar, optical = sar.to(device), optical.to(device)
            outputs = model(sar, optical)
            _, logs = criterion(outputs)
            for key, value in logs.items():
                totals[key] = totals.get(key, 0.0) + value
    return {key: value / max(len(loader), 1) for key, value in totals.items()}


def fit_publication_model(model, train_loader, val_loader, optimizer, criterion, device,
                          epochs=None, output_dir='publication_runs', resume=False):
    epochs = int(epochs or config.epochs)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    last_path = output_path / 'last_checkpoint.pt'
    best_path = output_path / 'best_checkpoint.pt'
    history = []
    start_epoch = 0
    best_val_loss = float('inf')

    if resume and last_path.exists():
        checkpoint = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        history = checkpoint.get('history', [])
        print(f'Resuming from epoch {start_epoch + 1}')

    for epoch in range(start_epoch, epochs):
        train_logs = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_logs = evaluate_one_epoch(model, val_loader, criterion, device)
        record = {'epoch': epoch + 1,
                  **{f'train_{k}': v for k, v in train_logs.items()},
                  **{f'val_{k}': v for k, v in val_logs.items()}}
        history.append(record)
        print(f"Epoch {epoch + 1:03d}/{epochs:03d} | train={train_logs['loss']:.5f} | val={val_logs['loss']:.5f}")

        checkpoint = {'epoch': epoch, 'model': model.state_dict(),
                      'optimizer': optimizer.state_dict(),
                      'best_val_loss': best_val_loss, 'history': history,
                      'config': asdict(config)}
        torch.save(checkpoint, last_path)
        if val_logs['loss'] < best_val_loss:
            best_val_loss = val_logs['loss']
            checkpoint['best_val_loss'] = best_val_loss
            torch.save(checkpoint, best_path)
            print(f"  saved new best checkpoint: {best_val_loss:.5f}")

    with open(output_path / 'history.json', 'w', encoding='utf-8') as handle:
        json.dump(history, handle, indent=2)
    return history

# One epoch is only a smoke test. For the paper, run the full configured training.
# history = fit_publication_model(
#     model, train_loader, val_loader, optimizer, criterion, device,
#     epochs=config.epochs, output_dir=config.output_dir, resume=False
# )

In [ ]:
history = fit_publication_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=100,
    output_dir="publication_runs",
    resume=True
)

In [ ]:
(history.json).head()

In [ ]:
# FINAL HELD-OUT TEST EVALUATION - run only after the 100-epoch training has finished.
# It loads the best validation checkpoint and creates NEW test-set FAISS indexes.
# Do not use the old precomputed gallery indexes from the prototype.

import csv
import faiss
from collections import defaultdict


@torch.no_grad()
def deterministic_hash_codes(model, loader, modality, device):
    """Create repeatable 192-bit codes for exactly one sensor modality.

    Training uses random masks; evaluation uses one fixed disjoint mask so that
    a given image always receives the same hash code.
    """
    model.eval()
    num_patches = model.num_patches
    target_count = int(num_patches * model.config.target_ratio)
    target_positions = torch.arange(target_count, device=device).unsqueeze(0)
    source_visible = torch.ones(num_patches, dtype=torch.bool, device=device)
    source_visible[target_count:2 * target_count] = False
    all_codes = []

    for sar, optical in loader:
        sar, optical = sar.to(device, non_blocking=True), optical.to(device, non_blocking=True)
        batch_size = sar.size(0)
        positions = target_positions.expand(batch_size, -1)
        visible = source_visible.unsqueeze(0).expand(batch_size, -1)

        if modality == 'optical':
            tokens = model.online_encoders.s2_encoder.forward_features(optical)[:, 1:]
            predicted = model.optical_to_sar(select_tokens(tokens, visible), positions)
        elif modality == 'sar':
            tokens = model.online_encoders.s1_encoder.forward_features(sar)[:, 1:]
            predicted = model.sar_to_optical(select_tokens(tokens, visible), positions)
        else:
            raise ValueError("modality must be 'optical' or 'sar'")

        all_codes.append(model.hash_head(predicted.mean(dim=1)).cpu().numpy())

    continuous = np.ascontiguousarray(np.concatenate(all_codes, axis=0), dtype=np.float32)
    return np.ascontiguousarray(np.packbits(continuous > 0, axis=1))


def _mean_metric(values):
    return float(np.mean(values)) if values else float('nan')


def evaluate_binary_retrieval(query_codes, gallery_codes, query_labels, gallery_labels,
                              k=10, exclude_identical_index=False, paired_retrieval=False):
    """Evaluate semantic-label retrieval; optional paired score is for cross-modal pairs."""
    if len(query_codes) != len(query_labels) or len(gallery_codes) != len(gallery_labels):
        raise ValueError('Every code must have matching labels.')
    index = faiss.IndexBinaryFlat(query_codes.shape[1] * 8)
    index.add(np.ascontiguousarray(gallery_codes))
    search_k = min(len(gallery_codes), k + int(exclude_identical_index))
    started = time.perf_counter()
    distances, retrieved = index.search(np.ascontiguousarray(query_codes), search_k)
    elapsed_ms = (time.perf_counter() - started) * 1000 / max(len(query_codes), 1)

    label_to_gallery = defaultdict(set)
    for gallery_index, labels in enumerate(gallery_labels):
        for label in labels:
            label_to_gallery[str(label)].add(gallery_index)

    precision, recall, f1, ap, ndcg, paired_hits = [], [], [], [], [], []
    for query_index, labels in enumerate(query_labels):
        query_set = {str(label) for label in labels}
        relevant_all = set().union(*(label_to_gallery[label] for label in query_set)) if query_set else set()
        if exclude_identical_index and query_index < len(gallery_codes):
            relevant_all.discard(query_index)

        ranks = [int(item) for item in retrieved[query_index] if not (exclude_identical_index and item == query_index)][:k]
        relevant = np.array([item in relevant_all for item in ranks], dtype=bool)
        hits = int(relevant.sum())
        p_at_k = hits / max(k, 1)
        r_at_k = hits / max(len(relevant_all), 1)
        precision.append(p_at_k)
        recall.append(r_at_k)
        f1.append(2 * p_at_k * r_at_k / max(p_at_k + r_at_k, 1e-12))
        cumulative = np.cumsum(relevant)
        rank_positions = np.arange(1, len(relevant) + 1)
        ap.append(float((cumulative / rank_positions * relevant).sum() / max(min(len(relevant_all), k), 1)))
        discounts = 1.0 / np.log2(np.arange(2, len(relevant) + 2))
        ideal = discounts[:min(len(relevant_all), k)].sum()
        ndcg.append(float((relevant * discounts).sum() / max(ideal, 1e-12)))
        if paired_retrieval:
            paired_hits.append(float(query_index in ranks))

    metrics = {
        f'precision@{k}': _mean_metric(precision), f'recall@{k}': _mean_metric(recall),
        f'f1@{k}': _mean_metric(f1), f'map@{k}': _mean_metric(ap),
        f'ndcg@{k}': _mean_metric(ndcg), 'search_ms_per_query': elapsed_ms,
        'queries': int(len(query_codes)), 'gallery_size': int(len(gallery_codes)),
        'hash_bits': int(query_codes.shape[1] * 8),
        'bytes_per_image': int(query_codes.shape[1])
    }
    if paired_retrieval:
        metrics[f'paired_recall@{k}'] = _mean_metric(paired_hits)
    return metrics, index


def run_final_test_evaluation(config, device, batch_size=64, k_values=(1, 5, 10)):
    # Test split was never used for optimisation or checkpoint selection.
    test_dataset = BEN14KDataset(config.data_dir, config.parquet_file, split='test')
    if len(test_dataset) == 0:
        raise RuntimeError('No test pairs found. Check the data directory and metadata split names.')
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=0, pin_memory=torch.cuda.is_available())

    best_path = Path(config.output_dir) / 'best_checkpoint.pt'
    if not best_path.exists():
        raise FileNotFoundError(f'Missing {best_path}; finish training before evaluation.')
    checkpoint = torch.load(best_path, map_location=device, weights_only=False)
    test_model = StablePublicationAntarikshJEPA(config).to(device)
    test_model.load_state_dict(checkpoint['model'])
    test_model.eval()
    print(f"Loaded best checkpoint from epoch {checkpoint['epoch'] + 1}.")

    optical_codes = deterministic_hash_codes(test_model, test_loader, 'optical', device)
    sar_codes = deterministic_hash_codes(test_model, test_loader, 'sar', device)
    labels = list(test_dataset.labels)

    results, indexes = {}, {}
    modes = {
        'optical_to_sar': (optical_codes, sar_codes, False, True),
        'sar_to_optical': (sar_codes, optical_codes, False, True),
        'optical_to_optical': (optical_codes, optical_codes, True, False),
        'sar_to_sar': (sar_codes, sar_codes, True, False),
    }
    for k in k_values:
        for name, (queries, gallery, exclude_self, paired) in modes.items():
            metrics, index = evaluate_binary_retrieval(queries, gallery, labels, labels, k, exclude_self, paired)
            results.setdefault(name, {}).update(metrics)
            indexes[name] = index
            print(f"{name:22s} | P@{k}={metrics[f'precision@{k}']:.4f} | R@{k}={metrics[f'recall@{k}']:.4f} | mAP@{k}={metrics[f'map@{k}']:.4f}")

    evaluation_dir = Path(config.output_dir) / 'final_test_evaluation'
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    faiss.write_index_binary(indexes['optical_to_sar'], str(evaluation_dir / 'sar_test_192bit.index'))
    faiss.write_index_binary(indexes['sar_to_optical'], str(evaluation_dir / 'optical_test_192bit.index'))
    pd.DataFrame([{'task': task, **metrics} for task, metrics in results.items()]).to_csv(
        evaluation_dir / 'metrics.csv', index=False
    )
    with open(evaluation_dir / 'metrics.json', 'w', encoding='utf-8') as handle:
        json.dump(results, handle, indent=2)
    pd.DataFrame({'patch_id': test_dataset.patch_ids, 'labels': labels}).to_pickle(
        evaluation_dir / 'test_gallery_metadata.pkl'
    )
    print(f'Final report and fresh test indexes saved to: {evaluation_dir.resolve()}')
    return results


# Run this once after the training cells. It performs inference only; it does not retrain.
# final_test_results = run_final_test_evaluation(config, device)


In [ ]:
final_test_results = run_final_test_evaluation(config, device)

In [ ]:
# REPORT ARTIFACT GENERATOR — run after final_test_evaluation has completed.
# This cell DOES NOT retrain or alter the checkpoint. It only creates report-ready
# figures/tables from the saved history and held-out-test metrics.

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def wilson_interval(successes, trials, z=1.96):
    """95% Wilson confidence interval for a proportion."""
    if trials <= 0:
        return float('nan'), float('nan')
    p = successes / trials
    denominator = 1 + z**2 / trials
    centre = (p + z**2 / (2 * trials)) / denominator
    half_width = z * np.sqrt(p * (1 - p) / trials + z**2 / (4 * trials**2)) / denominator
    return float(max(0, centre - half_width)), float(min(1, centre + half_width))


def generate_report_artifacts(output_dir=None):
    """Create transparent, publication-report evidence without re-training.

    Includes the exact-pair random baseline k/N. Semantic label-overlap metrics
    must be compared only against separately trained/implemented baselines.
    """
    if output_dir is None:
        output_dir = getattr(globals().get('config', None), 'output_dir', 'publication_runs')
    run_dir = Path(output_dir)
    metrics_path = run_dir / 'final_test_evaluation' / 'metrics.json'
    history_path = run_dir / 'history.json'
    if not metrics_path.exists():
        raise FileNotFoundError(f'Missing {metrics_path}. Run final_test_evaluation first.')
    if not history_path.exists():
        raise FileNotFoundError(f'Missing {history_path}. Keep the training history file with the checkpoint.')

    report_dir = run_dir / 'report_artifacts'
    report_dir.mkdir(parents=True, exist_ok=True)
    with open(metrics_path, encoding='utf-8') as handle:
        metrics = json.load(handle)
    with open(history_path, encoding='utf-8') as handle:
        history = json.load(handle)
    history_df = pd.DataFrame(history)

    fig, ax = plt.subplots(figsize=(8, 4.5), dpi=180)
    ax.plot(history_df['epoch'], history_df['train_loss'], label='Training loss', linewidth=1.7)
    ax.plot(history_df['epoch'], history_df['val_loss'], label='Validation loss', linewidth=1.7)
    best_row = history_df.loc[history_df['val_loss'].idxmin()]
    ax.scatter([best_row['epoch']], [best_row['val_loss']], color='#c43c39', zorder=3, label='Best validation checkpoint')
    ax.set(xlabel='Epoch', ylabel='Loss', title='Training and validation convergence')
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(report_dir / 'training_validation_loss.png', bbox_inches='tight')
    plt.show()
    plt.close(fig)

    paired_rows = []
    for task in ('optical_to_sar', 'sar_to_optical'):
        task_metrics = metrics[task]
        n = int(task_metrics['queries'])
        gallery_size = int(task_metrics['gallery_size'])
        for k in (1, 5, 10):
            observed = float(task_metrics[f'paired_recall@{k}'])
            successes = int(round(observed * n))
            ci_low, ci_high = wilson_interval(successes, n)
            random_expected = min(k / gallery_size, 1.0)
            paired_rows.append({
                'task': task, 'k': k, 'paired_recall': observed, 'paired_hits': successes,
                'queries': n, 'random_expected_recall': random_expected,
                'uplift_over_random': observed / random_expected if random_expected else float('nan'),
                'wilson_95_low': ci_low, 'wilson_95_high': ci_high,
            })
    paired_df = pd.DataFrame(paired_rows)
    paired_df.to_csv(report_dir / 'paired_retrieval_vs_random.csv', index=False)

    semantic_rows = []
    for task, task_metrics in metrics.items():
        semantic_rows.append({
            'task': task, 'P@1': task_metrics['precision@1'], 'mAP@5': task_metrics['map@5'],
            'mAP@10': task_metrics['map@10'], 'nDCG@10': task_metrics['ndcg@10'],
            'hash_bits': task_metrics['hash_bits'], 'bytes_per_image': task_metrics['bytes_per_image'],
            'search_ms_per_query': task_metrics['search_ms_per_query'],
        })
    semantic_df = pd.DataFrame(semantic_rows)
    semantic_df.to_csv(report_dir / 'semantic_retrieval_summary.csv', index=False)

    summary = {
        'protocol': {
            'checkpoint_selection': 'best validation loss; test set used only for final evaluation',
            'test_queries_and_gallery': int(metrics['optical_to_sar']['queries']),
            'hash_code': f"{metrics['optical_to_sar']['hash_bits']} bits / {metrics['optical_to_sar']['bytes_per_image']} bytes per item",
            'primary_metric_interpretation': 'semantic land-cover label-overlap retrieval',
            'secondary_diagnostic': 'exact paired optical-SAR tile retrieval',
        },
        'best_validation': {'epoch': int(best_row['epoch']), 'loss': float(best_row['val_loss'])},
        'important_limitation': 'Exact paired Recall@k is only modestly above the k/N random expectation. Do not claim state-of-the-art cross-modal instance matching from these results.',
        'next_required_experiments': [
            'Random binary-code baseline for every reported metric',
            'Original one-way prototype baseline under the identical test protocol',
            'Ablation: remove PSA, remove Soft-HGR, remove hash loss, and remove bidirectional training',
            'Three independent training seeds with mean and standard deviation',
            'A larger, disjoint gallery scalability study with index memory and end-to-end latency',
        ],
    }
    with open(report_dir / 'report_readiness_summary.json', 'w', encoding='utf-8') as handle:
        json.dump(summary, handle, indent=2)

    print(f'Report artifacts saved to: {report_dir.resolve()}')
    print(f"Best validation checkpoint: epoch {int(best_row['epoch'])}, loss={best_row['val_loss']:.5f}")
    print('Paired exact-retrieval context (not the semantic metric):')
    display(paired_df.round(6))
    print('Semantic retrieval summary:')
    display(semantic_df.round(6))
    return semantic_df, paired_df, summary


# Run once. It is safe to re-run; it overwrites only report_artifacts files.
# semantic_report, paired_report, report_summary = generate_report_artifacts()
semantic_report, paired_report, report_summary = generate_report_artifacts(
    "publication_runs"
)

In [ ]:
# RANDOM BINARY BASELINE — run after the FINAL HELD-OUT TEST EVALUATION cell.
# No checkpoint is loaded and no training occurs. This is the required chance-level
# comparator for the exact same test labels, gallery size, hash length, and metrics.

from pathlib import Path
import numpy as np
import pandas as pd


def run_random_binary_baseline(output_dir=None, seeds=(11, 22, 33, 44, 55), k_values=(1, 5, 10)):
    if 'evaluate_binary_retrieval' not in globals():
        raise RuntimeError('Run the FINAL HELD-OUT TEST EVALUATION definition cell first.')
    if output_dir is None:
        output_dir = getattr(globals().get('config', None), 'output_dir', 'publication_runs')
    run_dir = Path(output_dir)
    metadata_path = run_dir / 'final_test_evaluation' / 'test_gallery_metadata.pkl'
    if not metadata_path.exists():
        raise FileNotFoundError(f'Missing {metadata_path}. Run final_test_evaluation first.')

    metadata = pd.read_pickle(metadata_path)
    labels = list(metadata['labels'])
    n_items = len(labels)
    hash_bytes = 24  # Must match the trained 192-bit retrieval representation.
    records = []

    for seed in seeds:
        rng = np.random.default_rng(seed)
        # Independent random codes for each modality: a true no-learning baseline.
        optical_codes = rng.integers(0, 256, size=(n_items, hash_bytes), dtype=np.uint8)
        sar_codes = rng.integers(0, 256, size=(n_items, hash_bytes), dtype=np.uint8)
        for k in k_values:
            for task, query_codes, gallery_codes in (
                ('optical_to_sar', optical_codes, sar_codes),
                ('sar_to_optical', sar_codes, optical_codes),
            ):
                metrics, _ = evaluate_binary_retrieval(
                    query_codes, gallery_codes, labels, labels, k=k,
                    exclude_identical_index=False, paired_retrieval=True,
                )
                records.append({'seed': seed, 'task': task, 'k': k, **metrics})

    raw = pd.DataFrame(records)
    metric_columns = ['precision@1', 'precision@5', 'precision@10', 'map@1', 'map@5', 'map@10',
                      'ndcg@1', 'ndcg@5', 'ndcg@10', 'paired_recall@1', 'paired_recall@5', 'paired_recall@10']
    present_metrics = [name for name in metric_columns if name in raw.columns]
    summary = raw.groupby(['task', 'k'])[present_metrics].agg(['mean', 'std']).reset_index()
    summary.columns = ['_'.join(filter(None, map(str, column))).rstrip('_') for column in summary.columns]

    report_dir = run_dir / 'report_artifacts'
    report_dir.mkdir(parents=True, exist_ok=True)
    raw.to_csv(report_dir / 'random_binary_baseline_raw.csv', index=False)
    summary.to_csv(report_dir / 'random_binary_baseline_summary.csv', index=False)
    print(f'Random-baseline results saved to: {report_dir.resolve()}')
    display(summary.round(6))
    return raw, summary


# Run once. Expected runtime: a few minutes on CPU; no GPU/training needed.
random_baseline_raw, random_baseline_summary = run_random_binary_baseline(
    "publication_runs"
)

In [ ]:
# CONTRASTIVE RETRIEVAL-AWARE EXPERIMENT
# Run only AFTER the original publication run and its final evaluation are saved.
# This creates a completely NEW run directory. It does not overwrite publication_runs.

from dataclasses import replace
import torch.nn.functional as F


def symmetric_cross_modal_infonce(sar_codes, optical_codes, temperature=0.10):
    """Align each SAR/optical pair and repel every non-matching batch pair.

    For a batch of B aligned tiles, the positive for item i is item i in the
    other modality. All B-1 other items are explicit negatives. The objective is
    (CE(SAR->Optical) + CE(Optical->SAR)) / 2.
    """
    if sar_codes.size(0) < 2:
        raise ValueError('InfoNCE needs batch_size >= 2.')
    sar_norm = F.normalize(sar_codes, dim=1)
    optical_norm = F.normalize(optical_codes, dim=1)
    logits = sar_norm @ optical_norm.T / temperature
    targets = torch.arange(logits.size(0), device=logits.device)
    return 0.5 * (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets))


class ContrastivePublicationLoss(PublicationLoss):
    """Original JEPA objective plus instance-level cross-modal retrieval loss."""
    def __init__(self, config, lambda_contrastive=0.25, temperature=0.10):
        super().__init__(config)
        self.lambda_contrastive = float(lambda_contrastive)
        self.temperature = float(temperature)

    def forward(self, outputs):
        base_total, logs = super().forward(outputs)
        contrastive = symmetric_cross_modal_infonce(
            outputs['sar_hash'], outputs['optical_hash'], self.temperature
        )
        total = base_total + self.lambda_contrastive * contrastive
        logs['contrastive'] = contrastive.detach().item()
        logs['loss'] = total.detach().item()
        return total, logs


def setup_contrastive_experiment(base_config, output_dir='publication_runs_contrastive',
                                 lambda_contrastive=0.25, temperature=0.10):
    """Prepare a fresh model/optimizer; the existing trained model is never reused."""
    contrastive_config = replace(base_config, output_dir=output_dir)
    set_reproducibility(contrastive_config.seed)
    contrastive_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    contrastive_model = StablePublicationAntarikshJEPA(contrastive_config).to(contrastive_device)
    contrastive_criterion = ContrastivePublicationLoss(
        contrastive_config, lambda_contrastive=lambda_contrastive, temperature=temperature
    ).to(contrastive_device)
    contrastive_optimizer = torch.optim.AdamW(
        contrastive_model.parameters(), lr=contrastive_config.learning_rate,
        weight_decay=contrastive_config.weight_decay
    )
    print(f'Fresh contrastive run ready: {Path(output_dir).resolve()}')
    print(f'InfoNCE: lambda={lambda_contrastive}, temperature={temperature}, batch negatives={contrastive_config.batch_size - 1}')
    return contrastive_config, contrastive_device, contrastive_model, contrastive_criterion, contrastive_optimizer


# STEP 1 — execute this line once. Existing model/checkpoint remains untouched.
contrastive_config, contrastive_device, contrastive_model, contrastive_criterion, contrastive_optimizer = setup_contrastive_experiment(config)

In [ ]:
# STEP 2 — execute after step 1. This is a NEW 100-epoch training run (~same time as before).
contrastive_history = fit_publication_model(
    contrastive_model, train_loader, val_loader, contrastive_optimizer, contrastive_criterion,
    contrastive_device, epochs=contrastive_config.epochs,
    output_dir=contrastive_config.output_dir, resume=True
)

In [ ]:
# STEP 3 — after training, perform the same untouched held-out test evaluation.
contrastive_test_results = run_final_test_evaluation(contrastive_config, contrastive_device)

# STEP 4 — create report artifacts and the matching random baseline for this new run.
semantic_report_c, paired_report_c, report_summary_c = generate_report_artifacts(contrastive_config.output_dir)
random_raw_c, random_summary_c = run_random_binary_baseline(contrastive_config.output_dir)


In [ ]:
# ORIGINAL VS CONTRASTIVE COMPARISON — inference-free report table and figure.
# Run after final_test_evaluation for both publication_runs and publication_runs_contrastive.

import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


def compare_publication_runs(original_dir='publication_runs', contrastive_dir='publication_runs_contrastive'):
    original_dir, contrastive_dir = Path(original_dir), Path(contrastive_dir)
    paths = {
        'Original JEPA': original_dir / 'final_test_evaluation' / 'metrics.json',
        'Contrastive JEPA': contrastive_dir / 'final_test_evaluation' / 'metrics.json',
    }
    for name, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f'Missing {name} test metrics: {path}')
    loaded = {name: json.loads(path.read_text(encoding='utf-8')) for name, path in paths.items()}

    rows = []
    for task in ('optical_to_sar', 'sar_to_optical'):
        original, contrastive = loaded['Original JEPA'][task], loaded['Contrastive JEPA'][task]
        for metric in ('precision@1', 'map@5', 'map@10', 'ndcg@10', 'paired_recall@1', 'paired_recall@5', 'paired_recall@10'):
            before, after = float(original[metric]), float(contrastive[metric])
            rows.append({
                'task': task, 'metric': metric, 'original_jepa': before,
                'contrastive_jepa': after, 'absolute_change': after - before,
                'relative_multiplier': after / before if before else float('nan'),
            })
    comparison = pd.DataFrame(rows)
    output_dir = contrastive_dir / 'report_artifacts' / 'original_vs_contrastive'
    output_dir.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(output_dir / 'cross_modal_comparison.csv', index=False)

    # Compact paper figure: only the two cross-modal directions and most important metrics.
    figure_metrics = [('map@10', 'mAP@10'), ('paired_recall@10', 'Exact paired Recall@10')]
    tasks = ['optical_to_sar', 'sar_to_optical']
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=180)
    x = np.arange(len(tasks))
    width = 0.34

    lines_labels = []
    
    for ax, (metric, label) in zip(axes, figure_metrics):
        before = [loaded['Original JEPA'][task][metric] for task in tasks]
        after = [loaded['Contrastive JEPA'][task][metric] for task in tasks]
        b_bars = ax.bar(x - width / 2, before, width, label='Original JEPA')
        a_bars = ax.bar(x + width / 2, after, width, label='Contrastive JEPA')
        if not lines_labels:
            lines_labels = [b_bars, a_bars]
            
        ax.set_xticks(x, ['Optical→SAR', 'SAR→Optical'])
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.grid(axis='y', alpha=0.25)
        
        max_val = max(max(before), max(after))
        ax.set_ylim(0, max_val * 1.15)
        
        for bars in ax.containers:
            ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
            
    fig.legend(
        [lines_labels[0], lines_labels[1]], 
        ['Original JEPA', 'Contrastive JEPA'], 
        loc='upper center', 
        bbox_to_anchor=(0.5, 0.92), 
        ncol=2, 
        frameon=False
    )
            
    fig.suptitle('Held-out cross-modal retrieval: original vs contrastive objective')
    fig.tight_layout(rect=[0, 0, 1, 0.88])
    fig.savefig(output_dir / 'original_vs_contrastive_cross_modal.png', bbox_inches='tight')
    plt.show()
    plt.close(fig)

    print(f'Comparison artifacts saved to: {output_dir.resolve()}')
    display(comparison.round(6))
    return comparison


comparison_table = compare_publication_runs(
    r"CHANGE_ME_RUNS_PARENT\publication_runs",
    r"CHANGE_ME_RUNS_PARENT\publication_runs_contrastive"
)


In [ ]:
# REPORT FIGURE 2: DATASET AND PREPROCESSING
# This cell does NOT train or modify any checkpoint. It creates an original,
# report-ready figure from the exact local BEN-14K data used by this project.

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np


def create_dataset_preprocessing_figure(config_to_use=None, output_dir=None, sample_index=42):
    """Create Figure 2 from the actual paired dataset and preprocessing protocol."""
    if config_to_use is None:
        config_to_use = globals().get('contrastive_config', globals().get('config'))
    if config_to_use is None:
        raise RuntimeError('Run the PublicationConfig cell first.')
    if output_dir is None:
        output_dir = getattr(config_to_use, 'output_dir', 'publication_runs_contrastive')

    datasets = {
        'Train': BEN14KDataset(config_to_use.data_dir, config_to_use.parquet_file, split='train'),
        'Validation': BEN14KDataset(config_to_use.data_dir, config_to_use.parquet_file, split='validation'),
        'Test': BEN14KDataset(config_to_use.data_dir, config_to_use.parquet_file, split='test'),
    }
    test_data = datasets['Test']
    if not len(test_data):
        raise RuntimeError('No valid test pairs were found.')
    sample_index = min(max(int(sample_index), 0), len(test_data) - 1)
    sar, optical = test_data[sample_index]

    # Display-only optical composite: channel-wise min-max scaling avoids claiming
    # a calibrated true-colour product when source band ordering is unspecified.
    optical_preview = optical[:3].numpy().transpose(1, 2, 0)
    optical_preview = (optical_preview - optical_preview.min(axis=(0, 1), keepdims=True))
    optical_preview /= optical_preview.max(axis=(0, 1), keepdims=True) + 1e-8
    sar_preview = sar[0].numpy()
    sar_preview = (sar_preview - sar_preview.min()) / (sar_preview.max() - sar_preview.min() + 1e-8)

    figure, axes = plt.subplots(2, 2, figsize=(13.5, 8), dpi=220)
    figure.suptitle('Dataset Composition and Preprocessing for Antariksh-JEPA', fontsize=17, fontweight='bold')

    axes[0, 0].imshow(optical_preview)
    axes[0, 0].set_title('Paired Sentinel-2 optical sample\n(three-channel display after normalisation)', fontweight='bold')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(sar_preview, cmap='gray')
    axes[0, 1].set_title('Paired Sentinel-1 SAR sample\n(first SAR channel after normalisation)', fontweight='bold')
    axes[0, 1].axis('off')

    split_names, split_counts = list(datasets.keys()), [len(dataset) for dataset in datasets.values()]
    bars = axes[1, 0].bar(split_names, split_counts, color=['#2878b5', '#2c9c69', '#d87926'])
    axes[1, 0].set_title('Actual paired samples available on disk', fontweight='bold')
    axes[1, 0].set_ylabel('Number of paired Optical-SAR tiles')
    axes[1, 0].grid(axis='y', alpha=0.25)
    axes[1, 0].bar_label(bars, labels=[f'{count:,}' for count in split_counts], padding=3)

    axes[1, 1].axis('off')
    preprocessing_text = (
        'Input tensors\n'
        '• Optical: 120 × 120 × 10 channels\n'
        '• SAR: 120 × 120 × 2 channels\n\n'
        'Optical preprocessing\n'
        'x_opt = clip(x, 0, 10000) / 10000\n\n'
        'SAR preprocessing\n'
        'x_sar = [clip(x, -25, 0) + 25] / 25\n\n'
        'Output range: [0, 1] for both modalities\n'
        'Each image is later divided into 15 × 15 = 225 patches.'
    )
    axes[1, 1].text(0.04, 0.96, preprocessing_text, va='top', fontsize=11,
                    family='monospace', bbox=dict(boxstyle='round,pad=0.7', facecolor='#f5f7fa', edgecolor='#7b8794'))

    figure.text(0.5, 0.02,
                'Figure generated from the exact paired data and preprocessing procedure used for the reported contrastive experiment.',
                ha='center', fontsize=9)
    figure.tight_layout(rect=(0, 0.05, 1, 0.94))

    destination = Path(output_dir) / 'report_artifacts'
    destination.mkdir(parents=True, exist_ok=True)
    figure_path = destination / 'figure_2_dataset_preprocessing.png'
    figure.savefig(figure_path, bbox_inches='tight')
    plt.show()
    plt.close(figure)
    print(f'Figure 2 saved to: {figure_path.resolve()}')
    print({name: len(dataset) for name, dataset in datasets.items()})
    return figure_path, {name: len(dataset) for name, dataset in datasets.items()}


# Run once after config/foundation cells. It takes no GPU training time.
figure_2_path, dataset_split_counts = create_dataset_preprocessing_figure(
    contrastive_config,
    output_dir=contrastive_config.output_dir,
    sample_index=42,
)


In [ ]:
# REPORT FIGURE 3: PROPOSED METHODOLOGY AND RETRIEVAL
# Revision note: arrow labels and the long mask annotation are intentionally omitted to prevent text overlap.
# Run after the contrastive experiment/evaluation cells. This does NOT retrain or alter any checkpoint.
# It creates a report-only architecture figure, distinct from Figure 1 (high-level overview)
# and Figure 2 (dataset/preprocessing).

from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

figure_output_dir = Path(contrastive_config.output_dir) / 'report_artifacts'
figure_output_dir.mkdir(parents=True, exist_ok=True)
figure_3_path = figure_output_dir / 'figure_3_proposed_methodology_clean.png'
figure_3_svg_path = figure_output_dir / 'figure_3_proposed_methodology_clean.svg'

fig, ax = plt.subplots(figsize=(17, 10), dpi=220)
fig.patch.set_facecolor('#ffffff')
ax.set_facecolor('#ffffff')
ax.set_xlim(0, 17)
ax.set_ylim(0, 10)
ax.axis('off')

def box(x, y, w, h, title, body, face, edge='#1f2937', title_color='#111827'):
    patch = FancyBboxPatch(
        (x, y), w, h, boxstyle='round,pad=0.03,rounding_size=0.13',
        linewidth=1.4, edgecolor=edge, facecolor=face
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + h - 0.28, title, ha='center', va='top', fontsize=10.5,
            fontweight='bold', color=title_color)
    ax.text(x + w/2, y + h/2 - 0.12, body, ha='center', va='center', fontsize=8.3,
            color='#1f2937', linespacing=1.34, wrap=True)
    return (x, y, w, h)

def arrow(a, b, label=None, color='#475569', rad=0.0):
    x1, y1, w1, h1 = a
    x2, y2, w2, h2 = b
    start = (x1 + w1, y1 + h1/2)
    end = (x2, y2 + h2/2)
    art = FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=14,
                          linewidth=1.35, color=color,
                          connectionstyle=f'arc3,rad={rad}')
    ax.add_patch(art)
    if label:
        ax.text((start[0]+end[0])/2, (start[1]+end[1])/2 + 0.22, label,
                ha='center', va='bottom', fontsize=7.5, color=color, fontweight='bold')

def down_arrow(a, b, label=None, color='#475569'):
    x1, y1, w1, h1 = a
    x2, y2, w2, h2 = b
    start = (x1 + w1/2, y1)
    end = (x2 + w2/2, y2 + h2)
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=14,
                                 linewidth=1.35, color=color))
    if label:
        ax.text(start[0] + 0.1, (start[1]+end[1])/2, label, ha='left', va='center',
                fontsize=7.5, color=color, fontweight='bold')

# Header and phase bands
ax.text(8.5, 9.68, 'Antariksh-JEPA: Training Objective and Compact Retrieval Pipeline',
        ha='center', va='center', fontsize=17, fontweight='bold', color='#0f172a')
ax.text(8.5, 9.25, 'Training uses paired Optical–SAR tiles; retrieval stores only one 192-bit code per gallery image.',
        ha='center', va='center', fontsize=9.5, color='#475569')
ax.add_patch(FancyBboxPatch((0.25, 5.0), 16.5, 3.8, boxstyle='round,pad=0.04,rounding_size=0.12',
                            linewidth=1.2, edgecolor='#93c5fd', facecolor='#f8fbff'))
ax.text(0.55, 8.52, 'A. Self-supervised cross-modal representation learning (training only)',
        fontsize=10.5, fontweight='bold', color='#1d4ed8')

opt = box(0.65, 6.8, 2.25, 1.35, 'Optical tile', 'Sentinel-2\n10 × 120 × 120', '#dbeafe', '#2563eb')
sar = box(0.65, 5.2, 2.25, 1.35, 'SAR tile', 'Sentinel-1\n2 × 120 × 120', '#f3e8ff', '#7e22ce')
enc_o = box(3.45, 6.8, 2.35, 1.35, 'Online ViT encoder', '8 × 8 patches\n225 × 192 tokens', '#eff6ff', '#2563eb')
enc_s = box(3.45, 5.2, 2.35, 1.35, 'Online ViT encoder', '8 × 8 patches\n225 × 192 tokens', '#faf5ff', '#7e22ce')
pred = box(6.45, 6.15, 2.6, 1.65, 'Bidirectional predictors', 'O → S and S → O\n113 source context tokens\n→ 112 target-position vectors', '#ecfeff', '#0f766e')
target = box(9.7, 6.15, 2.55, 1.65, 'EMA target encoders', 'Full opposite modality\nselect the same 112\ntarget positions', '#fefce8', '#a16207')
loss = box(12.9, 6.15, 3.15, 1.65, 'Predictive objective', 'MSE + PSA + VICReg\n+ soft HGR + hash loss\n+ symmetric InfoNCE', '#fff7ed', '#c2410c')
arrow(opt, enc_o, color='#2563eb')
arrow(sar, enc_s, color='#7e22ce')
arrow(enc_o, pred, color='#2563eb', rad=-0.10)
arrow(enc_s, pred, color='#7e22ce', rad=0.10)
arrow(pred, target, color='#0f766e')
arrow(target, loss, color='#a16207')

ax.add_patch(FancyBboxPatch((0.25, 0.5), 16.5, 3.8, boxstyle='round,pad=0.04,rounding_size=0.12',
                            linewidth=1.2, edgecolor='#86efac', facecolor='#f7fff8'))
ax.text(0.55, 4.03, 'B. Compact cross-modal retrieval (held-out test / Streamlit application)',
        fontsize=10.5, fontweight='bold', color='#15803d')
query = box(0.65, 2.05, 2.25, 1.25, 'Query image', 'Optical or SAR\npreprocessed tile', '#f0fdf4', '#15803d')
query_enc = box(3.45, 2.05, 2.35, 1.25, 'Encoder + predictor', 'Online encoder +\ncross-modal predictor\n→ vector z ∈ ℝ¹⁹²', '#ecfdf5', '#15803d')
hash_q = box(6.45, 2.05, 2.5, 1.25, 'Binary hash head', 'Converts the continuous vector\ninto one 192-bit image code\n(192 bits = 24 bytes)', '#f0fdfa', '#0f766e')
index = box(9.35, 2.05, 2.45, 1.25, 'FAISS binary index', 'One packed 192-bit code\nper gallery image', '#f0fdf4', '#15803d')
rank = box(12.45, 2.05, 3.15, 1.25, 'Ranked retrieved images', 'Hamming distance\ncounts bit positions that differ\nbetween query code and gallery code', '#f0fdf4', '#15803d')
arrow(query, query_enc, color='#15803d')
arrow(query_enc, hash_q, color='#15803d')
arrow(hash_q, index, color='#15803d')
arrow(index, rank, color='#15803d')

ax.text(8.5, 1.24, 'At test time, losses and EMA target encoders are not used; the online encoder, cross-modal predictor, hash head, and binary index are required.',
        ha='center', va='center', fontsize=8.5, color='#166534', style='italic')
ax.text(8.5, 0.18, 'Source: Authors’ design.', ha='center', va='center', fontsize=8.5, color='#64748b')

fig.savefig(figure_3_path, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())
fig.savefig(figure_3_svg_path, format='svg', bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Clean Figure 3 PNG saved to: {figure_3_path}')
print(f'Clean Figure 3 editable SVG saved to: {figure_3_svg_path}')

In [ ]:
# JOURNAL-READINESS POST-TRAINING ARTIFACTS
# Safe to run AFTER the final held-out evaluation. It does NOT train, fine-tune,
# overwrite checkpoints, or modify FAISS indexes. It only reads saved artifacts
# and performs optional inference for a small qualitative panel.

from pathlib import Path
import hashlib
import json
import platform
import sys
import time
import textwrap

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def _resolve_existing_run_dir(path_or_name, fallback_name):
    """Resolve a saved run without hard-coding another machine's path."""
    candidates = []
    if path_or_name is not None:
        candidates.append(Path(path_or_name))
    candidates.extend([Path(fallback_name), Path.cwd() / fallback_name, Path.cwd().parent / fallback_name])
    for candidate in candidates:
        if (candidate / 'best_checkpoint.pt').exists() and (candidate / 'final_test_evaluation' / 'metrics.json').exists():
            return candidate.resolve()
    looked = '\n'.join(str(item) for item in candidates)
    raise FileNotFoundError(f'Could not locate saved run. Looked in:\n{looked}')


def _sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk_size), b''):
            digest.update(block)
    return digest.hexdigest()


def _wilson_interval(successes, trials, z=1.96):
    if trials <= 0:
        return float('nan'), float('nan')
    proportion = successes / trials
    denominator = 1.0 + z ** 2 / trials
    centre = (proportion + z ** 2 / (2.0 * trials)) / denominator
    half = z * np.sqrt(proportion * (1.0 - proportion) / trials + z ** 2 / (4.0 * trials ** 2)) / denominator
    return float(max(0.0, centre - half)), float(min(1.0, centre + half))


def _module_parameters(module):
    return int(sum(parameter.numel() for parameter in module.parameters()))


def create_journal_reproducibility_artifacts(
    original_run=None,
    contrastive_run=None,
    model=None,
    experiment_config=None,
):
    """Create no-training evidence for the journal report from saved outputs."""
    if experiment_config is None:
        experiment_config = globals().get('contrastive_config', globals().get('config'))
    if contrastive_run is None:
        contrastive_run = 'publication_runs_contrastive'
    if original_run is None:
        original_run = 'publication_runs'

    original_dir = _resolve_existing_run_dir(original_run, 'publication_runs')
    contrastive_dir = _resolve_existing_run_dir(contrastive_run, 'publication_runs_contrastive')
    output_dir = contrastive_dir / 'report_artifacts' / 'journal_readiness'
    output_dir.mkdir(parents=True, exist_ok=True)

    original_metrics = json.loads((original_dir / 'final_test_evaluation' / 'metrics.json').read_text(encoding='utf-8'))
    contrastive_metrics = json.loads((contrastive_dir / 'final_test_evaluation' / 'metrics.json').read_text(encoding='utf-8'))
    checkpoint_path = contrastive_dir / 'best_checkpoint.pt'
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    # Exact-pair Recall is a binomial event, so a Wilson 95% CI is valid without retraining.
    ci_rows = []
    for direction in ('optical_to_sar', 'sar_to_optical'):
        n = int(contrastive_metrics[direction]['queries'])
        for k in (1, 5, 10):
            for name, metrics in (('Original JEPA', original_metrics), ('Contrastive Antariksh-JEPA', contrastive_metrics)):
                recall = float(metrics[direction][f'paired_recall@{k}'])
                hits = int(round(recall * n))
                lower, upper = _wilson_interval(hits, n)
                ci_rows.append({
                    'retrieval_direction': direction,
                    'model': name,
                    'k': k,
                    'paired_hits': hits,
                    'queries': n,
                    'paired_recall': recall,
                    'paired_recall_percent': 100.0 * recall,
                    'wilson_95_ci_low_percent': 100.0 * lower,
                    'wilson_95_ci_high_percent': 100.0 * upper,
                })
    ci_df = pd.DataFrame(ci_rows)
    ci_df.to_csv(output_dir / 'paired_recall_wilson_95ci.csv', index=False)

    # Parameter accounting avoids the inaccurate claim that only the encoder is deployed.
    if model is None:
        model = globals().get('contrastive_model')
    if model is None and experiment_config is not None and 'StablePublicationAntarikshJEPA' in globals():
        model = StablePublicationAntarikshJEPA(experiment_config)
        model.load_state_dict(checkpoint['model'])
    parameter_summary = {'available': model is not None}
    if model is not None:
        parameter_summary.update({
            'training_model_total_parameters': _module_parameters(model),
            'training_model_trainable_parameters': int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
            'optical_to_sar_deployment_parameters': _module_parameters(model.online_encoders.s2_encoder) + _module_parameters(model.optical_to_sar) + _module_parameters(model.hash_head),
            'sar_to_optical_deployment_parameters': _module_parameters(model.online_encoders.s1_encoder) + _module_parameters(model.sar_to_optical) + _module_parameters(model.hash_head),
            'ema_target_encoder_parameters_excluded_at_inference': _module_parameters(model.target_encoders),
        })

    first_task = contrastive_metrics['optical_to_sar']
    manifest = {
        'purpose': 'post-training journal-report evidence; no training was performed by this cell',
        'run_directories': {'original': original_dir.name, 'contrastive': contrastive_dir.name},
        'checkpoint': {
            'relative_path': f'{contrastive_dir.name}/best_checkpoint.pt',
            'selected_epoch_one_based': int(checkpoint['epoch']) + 1,
            'sha256': _sha256(checkpoint_path),
        },
        'held_out_protocol': {
            'queries': int(first_task['queries']),
            'gallery_size': int(first_task['gallery_size']),
            'hash_bits': int(first_task['hash_bits']),
            'raw_hash_bytes_per_image': int(first_task['bytes_per_image']),
            'raw_hash_gallery_bytes': int(first_task['gallery_size']) * int(first_task['bytes_per_image']),
            'note': 'Raw hash storage excludes FAISS index overhead, gallery metadata, and model weights.',
        },
        'runtime': {
            'python': sys.version,
            'platform': platform.platform(),
            'torch': torch.__version__,
            'cuda_available': bool(torch.cuda.is_available()),
            'cuda_version': torch.version.cuda,
            'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        },
        'parameter_accounting': parameter_summary,
        'limitations_not_solved_by_this_cell': [
            'multi-seed training is still required for mean and standard deviation',
            'loss-component ablations require separate training runs',
            'external baseline reproduction requires separate implementations or checkpoints',
            'larger-gallery evaluation requires a new evaluation gallery',
        ],
    }
    (output_dir / 'journal_reproducibility_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

    result_rows = []
    for direction in ('optical_to_sar', 'sar_to_optical'):
        for metric in ('precision@1', 'map@10', 'ndcg@10', 'paired_recall@1', 'paired_recall@5', 'paired_recall@10'):
            result_rows.append({
                'retrieval_direction': direction,
                'metric': metric,
                'original_jepa': float(original_metrics[direction][metric]),
                'contrastive_antariksh_jepa': float(contrastive_metrics[direction][metric]),
                'absolute_change': float(contrastive_metrics[direction][metric] - original_metrics[direction][metric]),
            })
    pd.DataFrame(result_rows).to_csv(output_dir / 'journal_primary_results.csv', index=False)

    print(f'Journal-readiness artifacts saved to: {output_dir.resolve()}')
    print(f"Selected checkpoint: epoch {manifest['checkpoint']['selected_epoch_one_based']}")
    print(f"Raw hash-only gallery storage: {manifest['held_out_protocol']['raw_hash_gallery_bytes']:,} bytes")
    display(ci_df.round(6))
    return manifest, ci_df


def _percentile_rescale(image, low=2.0, high=98.0):
    """Contrast stretch only for report visualization; it never changes model input."""
    lo, hi = np.percentile(image, [low, high])
    return np.clip((image - lo) / max(hi - lo, 1e-6), 0.0, 1.0)


def _display_optical_like_app(tensor):
    """Same fixed-scale Sentinel-2 preview function used by Web_app/app.py."""
    array = tensor.detach().cpu().numpy()
    if array.shape[0] < 3:
        raise ValueError('Optical display requires at least three channels.')
    # BEN-14K's 10 optical bands begin B02, B03, B04; reorder to R, G, B.
    rgb = np.moveaxis(array[[2, 1, 0]], 0, -1)
    # Fixed reflectance scaling preserves RGB channel balance across all tiles.
    rgb = np.clip(rgb / 0.30, 0.0, 1.0)
    return np.power(rgb, 1.0 / 2.2)


def _display_sar_like_app(tensor):
    """Same Sentinel-1 VV grayscale preview function used by Web_app/app.py."""
    image = tensor[0].detach().cpu().numpy()
    return (image - image.min()) / (image.max() - image.min() + 1e-8)


def _format_semantic_labels(dataset, index, width=31, max_labels=2):
    """Short, readable BigEarthNet multi-label summary for a figure title."""
    labels = dataset.labels[index]
    if isinstance(labels, str):
        labels = [labels]
    labels = [str(label).replace('_', ' ') for label in labels]
    shown = labels[:max_labels]
    text = ', '.join(shown) + ('; …' if len(labels) > max_labels else '')
    return textwrap.fill(text or 'No metadata label', width=width)


@torch.no_grad()
def _selected_query_codes(model, dataset, indices, modality, device, batch_size=32):
    """Infer codes for a small fixed subset; this is inference only, not training."""
    model.eval()
    target_count = int(model.num_patches * model.config.target_ratio)
    positions_base = torch.arange(target_count, device=device).unsqueeze(0)
    visible_base = torch.ones(model.num_patches, dtype=torch.bool, device=device)
    visible_base[target_count:2 * target_count] = False
    codes = []
    for start in range(0, len(indices), batch_size):
        chunk = indices[start:start + batch_size]
        sar = torch.stack([dataset[index][0] for index in chunk]).to(device)
        optical = torch.stack([dataset[index][1] for index in chunk]).to(device)
        batch = len(chunk)
        positions = positions_base.expand(batch, -1)
        visible = visible_base.unsqueeze(0).expand(batch, -1)
        if modality == 'optical':
            tokens = model.online_encoders.s2_encoder.forward_features(optical)[:, 1:]
            predicted = model.optical_to_sar(select_tokens(tokens, visible), positions)
        elif modality == 'sar':
            tokens = model.online_encoders.s1_encoder.forward_features(sar)[:, 1:]
            predicted = model.sar_to_optical(select_tokens(tokens, visible), positions)
        else:
            raise ValueError("modality must be 'optical' or 'sar'")
        continuous = model.hash_head(predicted.mean(dim=1)).cpu().numpy()
        codes.append(np.packbits(continuous > 0, axis=1))
    return np.ascontiguousarray(np.concatenate(codes, axis=0))


def generate_qualitative_retrieval_panel(
    run_dir=None,
    experiment_config=None,
    candidate_count=256,
    top_k=3,
):
    """Generate Figure 6 from a small deterministic test subset; no training occurs.

    The panel intentionally includes an exact-pair success and a semantic-but-not-exact
    retrieval for each cross-modal direction. It must be described as qualitative evidence,
    not as an additional aggregate metric.
    """
    if experiment_config is None:
        experiment_config = globals().get('contrastive_config', globals().get('config'))
    if experiment_config is None:
        raise RuntimeError('Run the publication config cell first so the test dataset can be located.')
    run_dir = _resolve_existing_run_dir(run_dir or experiment_config.output_dir, 'publication_runs_contrastive')
    output_dir = run_dir / 'report_artifacts' / 'journal_readiness'
    output_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(run_dir / 'best_checkpoint.pt', map_location=device, weights_only=False)
    qualitative_model = StablePublicationAntarikshJEPA(experiment_config).to(device)
    qualitative_model.load_state_dict(checkpoint['model'])
    qualitative_model.eval()
    test_dataset = BEN14KDataset(experiment_config.data_dir, experiment_config.parquet_file, split='test')
    n = min(int(candidate_count), len(test_dataset))
    candidate_indices = list(range(n))
    selected_cases = []

    tasks = {
        'Optical → SAR': {
            'modality': 'optical',
            'index_path': run_dir / 'final_test_evaluation' / 'sar_test_192bit.index',
            'query_image': lambda idx: test_dataset[idx][1],
            'target_image': lambda idx: test_dataset[idx][0],
            'display_query': _display_optical_like_app,
            'display_target': _display_sar_like_app,
        },
        'SAR → Optical': {
            'modality': 'sar',
            'index_path': run_dir / 'final_test_evaluation' / 'optical_test_192bit.index',
            'query_image': lambda idx: test_dataset[idx][0],
            'target_image': lambda idx: test_dataset[idx][1],
            'display_query': _display_sar_like_app,
            'display_target': _display_optical_like_app,
        },
    }

    for direction, spec in tasks.items():
        index = faiss.read_index_binary(str(spec['index_path']))
        codes = _selected_query_codes(qualitative_model, test_dataset, candidate_indices, spec['modality'], device)
        distances, ranks = index.search(codes, top_k)
        records = []
        for local, query_index in enumerate(candidate_indices):
            retrieved = [int(item) for item in ranks[local]]
            exact_rank = retrieved.index(query_index) + 1 if query_index in retrieved else None
            top_index = retrieved[0]
            query_labels = {str(label) for label in test_dataset.labels[query_index]}
            top1_labels = {str(label) for label in test_dataset.labels[top_index]}
            semantic_top1 = bool(query_labels.intersection(top1_labels))
            records.append((query_index, retrieved, distances[local].tolist(), exact_rank, semantic_top1))
        success = next((record for record in records if record[3] is not None), None)
        semantic_non_exact = next((record for record in records if record[3] is None and record[4]), None)
        if success is None or semantic_non_exact is None:
            raise RuntimeError(f'Could not find both an exact-pair success and a semantic-but-non-exact top-1 case in the first {n} candidates for {direction}. Increase candidate_count.')
        selected_cases.extend([(direction, 'Exact pair in top-k', spec, success), (direction, 'Semantic but non-exact top-1', spec, semantic_non_exact)])

    fig, axes = plt.subplots(4, 3, figsize=(14, 16), dpi=220)
    rows = []
    for row, (direction, case_type, spec, record) in enumerate(selected_cases):
        query_index, retrieved, distances, exact_rank, semantic_top1 = record
        top_index = retrieved[0]
        query_labels = _format_semantic_labels(test_dataset, query_index)
        target_labels = _format_semantic_labels(test_dataset, top_index)
        axes[row, 0].imshow(spec['display_query'](spec['query_image'](query_index)), cmap='gray' if spec['modality'] == 'sar' else None)
        axes[row, 0].set_title(f'{direction} | {case_type}\nQuery test index: {query_index}\nQuery labels: {query_labels}', fontsize=9, pad=10)
        axes[row, 1].imshow(spec['display_target'](spec['target_image'](query_index)), cmap='gray' if spec['modality'] == 'optical' else None)
        axes[row, 1].set_title('True paired target\n(same geographical tile)', fontsize=9, pad=10)
        axes[row, 2].imshow(spec['display_target'](spec['target_image'](top_index)), cmap='gray' if spec['modality'] == 'optical' else None)
        axes[row, 2].set_title(f'Top-1 retrieved | Hamming: {int(distances[0])} | paired rank: {exact_rank or ">" + str(top_k)}\nTop-1 labels: {target_labels}', fontsize=9, pad=10)
        for column in range(3):
            axes[row, column].axis('off')
        rows.append({
            'direction': direction, 'case_type': case_type, 'query_index': query_index,
            'query_patch_id': test_dataset.patch_ids[query_index], 'top1_index': top_index,
            'top1_patch_id': test_dataset.patch_ids[top_index], 'top1_hamming_distance': int(distances[0]),
            'query_labels': ' | '.join(map(str, test_dataset.labels[query_index])),
            'top1_labels': ' | '.join(map(str, test_dataset.labels[top_index])),
            'paired_rank_within_top_k': exact_rank, 'top1_is_semantically_relevant': semantic_top1, 'top_k': top_k,
        })
    fig.suptitle('Qualitative held-out cross-modal retrieval examples', fontsize=15, fontweight='bold')
    fig.text(0.5, 0.01, 'Visual rendering exactly matches the local Streamlit application: Optical uses a fixed-scale, gamma-corrected B04/B03/B02 true-colour-style composite and SAR uses VV grayscale. Display processing never alters retrieval. Two short labels are shown for readability; complete BigEarthNet labels are saved in figure_6_case_metadata.csv. Cases are selected from a fixed candidate subset; use only as qualitative evidence.', ha='center', fontsize=8)
    fig.tight_layout(rect=[0, 0.035, 1, 0.97])
    png_path = output_dir / 'figure_6_qualitative_retrieval_examples.png'
    svg_path = output_dir / 'figure_6_qualitative_retrieval_examples.svg'
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(svg_path, format='svg', bbox_inches='tight')
    plt.show()
    plt.close(fig)
    pd.DataFrame(rows).to_csv(output_dir / 'figure_6_case_metadata.csv', index=False)
    print(f'Figure 6 PNG saved to: {png_path}')
    print(f'Figure 6 editable SVG saved to: {svg_path}')
    return pd.DataFrame(rows)


# Step A: run now. Reads existing checkpoints/metrics only; no inference or training.
journal_manifest, paired_recall_ci = create_journal_reproducibility_artifacts()

# Step B: optional; run later for Figure 6. It performs inference on only 256 held-out samples,
# not training. Increase candidate_count only if the cell asks you to.
figure_6_cases = generate_qualitative_retrieval_panel(candidate_count=256, top_k=3)

In [ ]:
# JOURNAL LATENCY BENCHMARK — no training; uses the saved contrastive checkpoint and test FAISS indexes.
# This full run measures 3,248 held-out queries in each cross-modal direction three times.
latency_summary, latency_per_query, latency_manifest = benchmark_retrieval_latency(
    warmup_queries=50, repeats=3, max_queries=None, top_k=10
)

In [ ]:
# SELF-CONTAINED JOURNAL LATENCY BENCHMARK — run this cell only; no training occurs.
import json, platform, time
from datetime import datetime, timezone

@torch.no_grad()
def run_self_contained_latency_benchmark(warmup_queries=50, repeats=3, max_queries=None, top_k=10):
    cfg = globals().get('contrastive_config', globals().get('config'))
    if cfg is None:
        raise RuntimeError('Run the publication configuration and contrastive setup cells first.')
    run_dir = _resolve_existing_run_dir(cfg.output_dir, 'publication_runs_contrastive')
    evaluation_dir = run_dir / 'final_test_evaluation'
    output_dir = run_dir / 'report_artifacts' / 'journal_readiness'
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / 'best_checkpoint.pt'
    index_paths = {'optical_to_sar': evaluation_dir / 'sar_test_192bit.index',
                   'sar_to_optical': evaluation_dir / 'optical_test_192bit.index'}
    if not checkpoint_path.exists() or not all(path.exists() for path in index_paths.values()):
        raise FileNotFoundError('Missing saved checkpoint or final-test FAISS index. Run final_test_evaluation first.')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = StablePublicationAntarikshJEPA(cfg).to(device)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    dataset = BEN14KDataset(cfg.data_dir, cfg.parquet_file, split='test')
    count = len(dataset) if max_queries is None else min(int(max_queries), len(dataset))
    if count < 1 or repeats < 1:
        raise ValueError('At least one query and one repeat are required.')
    tasks = {'optical_to_sar': ('optical', faiss.read_index_binary(str(index_paths['optical_to_sar']))),
             'sar_to_optical': ('sar', faiss.read_index_binary(str(index_paths['sar_to_optical'])))}
    target_count = int(model.num_patches * model.config.target_ratio)
    positions = torch.arange(target_count, device=device).unsqueeze(0)
    visible = torch.ones(model.num_patches, dtype=torch.bool, device=device)
    visible[target_count:2 * target_count] = False

    def timed_hash(image, modality):
        if device.type == 'cuda': torch.cuda.synchronize(device)
        started = time.perf_counter()
        image = image.unsqueeze(0).to(device, non_blocking=False)
        if modality == 'optical':
            tokens = model.online_encoders.s2_encoder.forward_features(image)[:, 1:]
            predicted = model.optical_to_sar(select_tokens(tokens, visible.unsqueeze(0)), positions)
        else:
            tokens = model.online_encoders.s1_encoder.forward_features(image)[:, 1:]
            predicted = model.sar_to_optical(select_tokens(tokens, visible.unsqueeze(0)), positions)
        code = np.ascontiguousarray(np.packbits(model.hash_head(predicted.mean(dim=1)).cpu().numpy() > 0, axis=1))
        if device.type == 'cuda': torch.cuda.synchronize(device)
        return code, (time.perf_counter() - started) * 1000.0

    print(f'Checkpoint epoch {checkpoint["epoch"] + 1} | device={device} | gallery={tasks["optical_to_sar"][1].ntotal}')
    print(f'Warm-up={warmup_queries} | measured={count} per repeat/direction | repeats={repeats}')
    for _, (modality, index) in tasks.items():
        for query_index in range(min(warmup_queries, count)):
            sar, optical = dataset[query_index]
            code, _ = timed_hash(optical if modality == 'optical' else sar, modality)
            index.search(code, min(top_k, index.ntotal))

    rows = []
    for repeat in range(1, repeats + 1):
        for direction, (modality, index) in tasks.items():
            for query_index in range(count):
                sar, optical = dataset[query_index]  # File loading is deliberately outside timing.
                code, inference_ms = timed_hash(optical if modality == 'optical' else sar, modality)
                started = time.perf_counter(); index.search(code, min(top_k, index.ntotal))
                search_ms = (time.perf_counter() - started) * 1000.0
                rows.append({'retrieval_direction': direction, 'repeat': repeat, 'query_index': query_index,
                             'encoder_predictor_hash_ms': inference_ms, 'faiss_binary_search_ms': search_ms,
                             'total_ms': inference_ms + search_ms})
            print(f'Completed {direction}, repeat {repeat}/{repeats}.')

    per_query = pd.DataFrame(rows)
    summary = per_query.groupby('retrieval_direction', as_index=False).agg(
        measured_queries=('query_index', 'count'),
        inference_mean_ms=('encoder_predictor_hash_ms', 'mean'), inference_std_ms=('encoder_predictor_hash_ms', 'std'),
        search_mean_ms=('faiss_binary_search_ms', 'mean'), search_std_ms=('faiss_binary_search_ms', 'std'),
        total_mean_ms=('total_ms', 'mean'), total_std_ms=('total_ms', 'std'),
        total_p50_ms=('total_ms', lambda x: float(np.percentile(x, 50))),
        total_p95_ms=('total_ms', lambda x: float(np.percentile(x, 95))))
    per_query.to_csv(output_dir / 'retrieval_latency_per_query.csv', index=False)
    summary.to_csv(output_dir / 'retrieval_latency_summary.csv', index=False)
    manifest = {'purpose': 'post-training latency benchmark; no training was performed',
                'checkpoint_epoch_one_based': int(checkpoint['epoch']) + 1, 'device': str(device),
                'gpu_name': torch.cuda.get_device_name(device) if device.type == 'cuda' else None,
                'torch_version': torch.__version__, 'platform': platform.platform(),
                'gallery_size': int(tasks['optical_to_sar'][1].ntotal), 'hash_bits': 192, 'top_k': int(top_k),
                'warmup_queries_per_direction': int(warmup_queries),
                'measured_queries_per_direction_per_repeat': int(count), 'repeats': int(repeats),
                'timing_scope': 'preprocessed tensor to device transfer, encoder, predictor, hash packing, and CPU FAISS search; excludes disk I/O and UI rendering',
                'created_utc': datetime.now(timezone.utc).isoformat()}
    (output_dir / 'retrieval_latency_benchmark.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    display(summary.round(4))
    print(f'Latency artifacts saved to: {output_dir}')
    return summary, per_query, manifest

latency_summary, latency_per_query, latency_manifest = run_self_contained_latency_benchmark(
    warmup_queries=50, repeats=3, max_queries=None, top_k=10
)